# Evaluación Final Transversal — Sistema multimodal de imágenes y texto

## Parte 2: Clasificación de sentimiento en reseñas de IMDB con redes recurrentes y Transformer (PyTorch)

**Institución:** Duoc UC, Escuela de Informática y Telecomunicaciones<br>
**Asignatura:** Deep Learning (DLY0100) · Sección 003D<br>
**Docente:** Guillermo Eugenio Pinto Fuentes<br>
**Integrantes:** Esteban Pacheco Vega, Francisco Figueroa Donoso<br>
**Fecha:** 25-11-2026<br>
**Repositorio:** https://github.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D

## 1. Introducción

### 1.1 Descripción del problema

El proyecto integra dos modalidades de datos, imágenes de prendas y reseñas de texto, para obtener una predicción conjunta. El trabajo se divide en tres notebooks: el primero entrena la red convolucional que clasifica las imágenes de Fashion-MNIST; **este entrena los modelos de texto** que clasifican el sentimiento de reseñas de películas; el tercero fusiona las representaciones de ambos en un clasificador final.

El dataset **IMDB** contiene 50.000 reseñas en inglés etiquetadas como positivas o negativas, exactamente 25.000 de cada clase. Las reseñas son largas y de largo muy variable (de 8 a más de 2.700 tokens, con una mediana cercana a 200), contienen etiquetas HTML de salto de línea (`<br />`) y algunas aparecen repetidas. Estas cifras se calculan en la sección 3.

El sentimiento de una reseña depende del **orden** de las palabras, no solo de cuáles aparecen: «*not good, but fun*» y «*not fun, but good*» usan exactamente las mismas palabras y su tono es distinto. Por eso se comparan dos familias de modelos que leen la secuencia completa:

- **Redes recurrentes** (RNN simple, LSTM y GRU): leen la reseña token a token y resumen lo leído en un estado oculto que se actualiza en cada paso.
- **Transformer:** procesa todos los tokens a la vez y, mediante **autoatención**, cada token se relaciona directamente con todos los demás, sin pasar por una cadena de estados.

### 1.2 Objetivo del modelo

Entrenar y optimizar clasificadores de sentimiento de ambas familias, compararlos con evidencia propia y entregar al notebook de fusión una **representación del texto** de 128 valores por reseña: el vector de la última capa oculta del mejor modelo en validación.

- Se optimizan la tasa de aprendizaje, el tamaño del lote y el número de épocas en ambas familias, además de la arquitectura y la regularización de cada una.
- **Métrica de decisión: F1 macro en validación**, la misma de los otros dos notebooks. Con clases balanceadas coincide casi con la exactitud, pero no premia a un modelo que favorece una de las dos clases.

**Nota de terminología:** *accuracy* = **exactitud** (proporción de aciertos); *precision* = **precisión** (de las reseñas que el modelo llama positivas, cuántas lo son); *recall* = **exhaustividad** (de las reseñas positivas reales, cuántas detecta); F1 = media armónica de precisión y exhaustividad. El promedio **macro** pondera igual a ambas clases y el **ponderado** según su cantidad de ejemplos.

### 1.3 Estructura del notebook

1. Introducción.
2. Configuración inicial, reutilización de resultados y herramientas.
3. Carga y exploración de los datos, partición y líneas base.
4. Preprocesamiento: vocabulario, codificación y lotes por largo.
5. Utilidades de entrenamiento, evaluación y experimentos.
6. Arquitecturas recurrente y Transformer, y configuraciones base.
7. a 13. **Parte A — Redes recurrentes:** modelo base, rango de error, tipo de celda, entrenamiento, arquitectura, regularización y modelo final.
14. a 20. **Parte B — Transformer:** modelo base, rango de error, forma de resumir la reseña (encoder, `[CLS]` o decoder), entrenamiento, arquitectura, regularización y modelo final.
21. Evaluación de ambos modelos finales en test.
22. Comparación entre la red recurrente y el Transformer.
23. Comparación entre las configuraciones base y finales.
24. Síntesis de los experimentos.
25. Modelo de texto para la fusión.
26. a 28. Conclusiones, conclusión final y referencias.
29. a 32. Anexos (gráficos para la presentación y herramientas), respaldo y descripción de las salidas.

**Criterio metodológico:**

- Todas las decisiones se toman con el conjunto de **validación**. El conjunto de **test** se usa una sola vez por familia, con su modelo final.
- Se cambia **un hiperparámetro a la vez** sobre la configuración vigente de cada familia; la variante elegida se incorpora y el experimento siguiente parte de ella. Cada familia lleva su propia cadena de decisiones.
- Antes de comparar se mide el **rango de error** entre reentrenamientos con 5 semillas, por separado para cada familia. Una diferencia menor o igual a ese rango es un **empate técnico**, y los empates se deciden según el contexto del problema, no solo por la métrica.
- Si la mejor variante queda en el borde del rango probado, el rango se amplía un paso en esa dirección.
- Ambas familias usan los mismos datos, vocabulario, largo máximo, *embedding* de entrada, capa de representación, métricas y régimen de épocas, para que la comparación sea justa.
- Al final de cada familia se verifican las combinaciones con interacción conocida y se compara el modelo final con el base usando las mismas semillas.

## 2. Configuración inicial

Se importan en un único bloque todas las librerías del notebook:

- **PyTorch** (`torch.nn`, `pack_padded_sequence`): define las redes recurrentes, el Transformer y el ciclo de entrenamiento.
- **NumPy / Pandas**: arreglos y tablas comparativas.
- **scikit-learn**: partición estratificada, métricas, la referencia lineal TF-IDF con regresión logística y PCA de la representación.
- **Plotly**: gráficos interactivos. El renderizador `vscode+colab` permite verlos tanto en Colab como en VS Code conectado a un servidor de Colab.
- **IPython.display**: tablas con `display`.
- **re / collections.Counter**: limpieza, tokenización y conteo del vocabulario.
- **io / urllib / os / json / shutil**: lectura del CSV desde el repositorio y respaldo.
- **copy / math / random / time**: copia de los pesos de la mejor época, escalas de la atención, semillas y tiempos.

`fijar_semilla(semilla)` fija las semillas de `random`, NumPy y PyTorch (CPU y GPU). No retorna nada.

In [ ]:
# ============================================================
# Importación de librerías (todos los imports del notebook van aquí)
# ============================================================
import copy
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from collections import Counter
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence
try:
    from google.colab import files
except ImportError:
    files = None

# ============================================================
# Reproducibilidad, dispositivo y visualización
# ============================================================
SEED = 42


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pio.renderers.default = "vscode+colab"
COLORES = ["#104281", "#3987E5", "#86B6EF"]
AMARILLO = "#F7B500"
print("PyTorch", torch.__version__, "| dispositivo:", device)

**Nota:** aun con semilla fija, el entrenamiento en GPU no es completamente determinista. Por eso, las secciones 8 y 15 miden ese margen de variación antes de comparar configuraciones.

**Requisito de ejecución:** activar GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*).

### 2.1 Reutilización de resultados ya obtenidos

Para no repetir cómputo, las métricas de cada corrida se guardan en `resultados_texto_guardados.json` y se reutilizan cuando la configuración y la semilla coinciden **exactamente**: la clave de cada corrida incluye todos los hiperparámetros, la semilla, el tope de épocas y la paciencia.

- Las corridas que necesitan el modelo entrenado o sus curvas (modelos base, tipo de celda, forma de resumir la reseña, épocas y modelos finales) no se pueden reutilizar solo con las métricas: además se guardan como `.pt`.
- Cada corrida nueva se imprime en la salida de su celda como una línea `CORRIDA_GUARDADA {...}`. Si la sesión se corta, el JSON se reconstruye desde el notebook guardado, sin montar Google Drive.
- `leer_archivo(ruta, binario)` busca el archivo en la carpeta de trabajo y, si no está, lo descarga de `URL_REPO` (la dirección *raw* del repositorio del grupo). Retorna el contenido, o `None` si no está en ninguno de los dos lugares.
- `guardar_corridas()` escribe en el JSON las corridas guardadas más las nuevas.
- `guardar_modelo_corrida(clave, modelo, historia, fila)` guarda esas corridas en `corridas_texto/corrida_texto_<código>.pt`, con los pesos del modelo, la historia por época y la fila de métricas; el código es un *hash* de la clave. `cargar_modelo_corrida(clave, modelo)` busca ese archivo en `corridas_texto/` y en la raíz, primero en la carpeta de trabajo y después en el repositorio, y si lo encuentra carga los pesos en `modelo` (con `weights_only=True`) y retorna `(fila, historia)`; si no, retorna `None`. Así, al volver a ejecutar, esas corridas tampoco se entrenan. Con `REUTILIZAR_MODELOS = False` se entrenan siempre.
- `descargar_resultados()` descarga al equipo el JSON y los `.pt` nuevos cada vez que termina una celda de código que agregó corridas, para no perderlos si la sesión de Colab se cierra. Se activa con `DESCARGAR_RESULTADOS = True`. El navegador pide permiso para descargas múltiples la primera vez y numera las copias repetidas del JSON (`(1)`, `(2)`, …): la vigente es la más reciente, porque cada una contiene todas las corridas anteriores. Para reutilizarlos, el JSON (sin el número) va en la raíz del repositorio y los `.pt` en `corridas_texto/` o en la raíz. Fuera de Colab (por ejemplo, en VS Code con kernel de Colab) la descarga puede no funcionar: en ese caso se imprime un aviso y los archivos quedan en la carpeta de trabajo.

In [ ]:
# ============================================================
# Carga de los resultados guardados
# ============================================================
REUTILIZAR_RESULTADOS = True
ARCHIVO_RESULTADOS = "resultados_texto_guardados.json"

URL_REPO = "https://raw.githubusercontent.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D/main/"


def leer_archivo(ruta, binario=False):
    if os.path.exists(ruta):
        with open(ruta, "rb") as f:
            datos = f.read()
    else:
        try:
            with urllib.request.urlopen(URL_REPO + urllib.parse.quote(ruta)) as r:
                datos = r.read()
        except urllib.error.HTTPError:
            return None
    return datos if binario else datos.decode("utf-8")


def guardar_corridas():
    with open(ARCHIVO_RESULTADOS, "w", encoding="utf-8") as f:
        json.dump({**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}, f, ensure_ascii=False, indent=1, default=float)


texto_resultados = leer_archivo(ARCHIVO_RESULTADOS) if REUTILIZAR_RESULTADOS else None
RESULTADOS_GUARDADOS = json.loads(texto_resultados) if texto_resultados else {}
CORRIDAS_NUEVAS = {}
REUTILIZAR_MODELOS = True
CARPETA_MODELOS = "corridas_texto"
DESCARGAR_RESULTADOS = True
MODELOS_NUEVOS = []
descargados = {"corridas": 0, "modelos": 0}


def nombre_modelo_corrida(clave):
    return f"corrida_texto_{hashlib.sha1(clave.encode('utf-8')).hexdigest()[:16]}.pt"


def guardar_modelo_corrida(clave, modelo, historia, fila):
    os.makedirs(CARPETA_MODELOS, exist_ok=True)
    ruta = f"{CARPETA_MODELOS}/{nombre_modelo_corrida(clave)}"
    metadatos = json.dumps({"clave": clave, "fila": fila, "historia": historia}, default=float)
    torch.save({"estado": modelo.state_dict(), "metadatos": metadatos}, ruta)
    MODELOS_NUEVOS.append(ruta)


def cargar_modelo_corrida(clave, modelo):
    nombre = nombre_modelo_corrida(clave)
    for ruta in [f"{CARPETA_MODELOS}/{nombre}", nombre]:
        datos = leer_archivo(ruta, binario=True)
        if datos is None:
            continue
        guardado = torch.load(io.BytesIO(datos), map_location=device, weights_only=True)
        metadatos = json.loads(guardado["metadatos"])
        if metadatos["clave"] != clave:
            continue
        modelo.load_state_dict(guardado["estado"])
        return metadatos["fila"], metadatos["historia"]
    return None


def descargar_resultados(resultado=None):
    if not DESCARGAR_RESULTADOS or len(CORRIDAS_NUEVAS) == descargados["corridas"] and len(MODELOS_NUEVOS) == descargados["modelos"]:
        return
    pendientes = [ARCHIVO_RESULTADOS] + MODELOS_NUEVOS[descargados["modelos"]:]
    descargados["corridas"], descargados["modelos"] = len(CORRIDAS_NUEVAS), len(MODELOS_NUEVOS)
    try:
        for ruta in pendientes:
            files.download(ruta)
    except Exception as error:
        print(f"Descarga automática no disponible ({type(error).__name__}): los archivos quedan en la carpeta de trabajo")


eventos = get_ipython().events
for funcion in [f for f in eventos.callbacks["post_run_cell"] if getattr(f, "__name__", "") in ("descargar_json", "descargar_resultados")]:
    eventos.unregister("post_run_cell", funcion)
eventos.register("post_run_cell", descargar_resultados)
print(f"Corridas guardadas disponibles: {len(RESULTADOS_GUARDADOS)}")

### 2.2 Herramientas: fortalezas y debilidades

| Herramienta | Fortaleza en este proyecto | Debilidad o costo |
|---|---|---|
| **PyTorch** | Control explícito del ciclo de entrenamiento, del relleno (`pack_padded_sequence`) y de las máscaras de atención; permite escribir el Transformer bloque a bloque y extraer la capa oculta que necesita la fusión. Es el mismo framework de la CNN | Más código que Keras; los errores de forma de los tensores aparecen en ejecución |
| **Google Colab con GPU T4** | Las redes recurrentes con cuDNN y la atención en GPU entrenan una época en segundos o pocos minutos; el equipo local no tiene GPU compatible con CUDA | Cuota de uso limitada y sesiones que se cortan; los archivos quedan en el servidor y hay que descargarlos |
| **Tokenización propia con expresiones regulares** | Transparente y reproducible, sin descargar modelos ni diccionarios externos | No separa contracciones ni reconoce emoticones; un tokenizador por subpalabras reduciría las palabras desconocidas |
| **Plotly** | Gráficos interactivos (zoom, valores al pasar el cursor) y exportables a HTML y PNG para la presentación | Los gráficos pesan más en el notebook que una imagen estática |
| **scikit-learn** | Métricas estándar y verificadas, partición estratificada y una referencia lineal rápida (TF-IDF con regresión logística) | Trabaja en CPU con NumPy: las predicciones deben copiarse desde la GPU |

## 3. Carga y exploración de los datos

### 3.1 Lectura del archivo

Las reseñas se leen del archivo `IMDB Dataset.csv` ubicado en la raíz del repositorio (columnas `review` y `sentiment`). Se usa este archivo, y no una versión ya tokenizada de una librería, para trabajar con el texto original y controlar la limpieza y la tokenización.

In [ ]:
# ============================================================
# Lectura del CSV de IMDB
# ============================================================
ARCHIVO_IMDB = "IMDB Dataset.csv"
texto_csv = leer_archivo(ARCHIVO_IMDB)
if texto_csv is None:
    raise FileNotFoundError(f"No se encontró {ARCHIVO_IMDB} ni en la carpeta de trabajo ni en el repositorio")
imdb_original = pd.read_csv(io.StringIO(texto_csv))
SENTIMIENTOS = ["negativa", "positiva"]
print("Forma:", imdb_original.shape, "| columnas:", list(imdb_original.columns))

### 3.2 Validación de los datos y duplicados

Antes de entrenar se verifica que no haya reseñas vacías ni etiquetas fuera de `positive` / `negative`, y se buscan reseñas con **exactamente el mismo texto**. Una reseña repetida puede quedar una copia en entrenamiento y otra en test: el modelo la «recuerda» y la métrica de test sube sin que el modelo generalice mejor. Por eso se conserva una sola copia de cada texto **antes** de partir los datos. También se revisa si alguna reseña repetida tiene etiquetas contradictorias.

In [ ]:
# ============================================================
# Validación y eliminación de duplicados
# ============================================================
repetidas = imdb_original[imdb_original.duplicated("review", keep=False)]
validacion = pd.DataFrame({
    "comprobación": ["reseñas", "nulas", "vacías", "etiquetas distintas", "filas con texto repetido",
                     "copias sobrantes", "textos repetidos con etiquetas contradictorias", "reseñas con <br />"],
    "resultado": [len(imdb_original), int(imdb_original.isna().sum().sum()),
                  int((imdb_original["review"].str.strip() == "").sum()), sorted(imdb_original["sentiment"].unique()),
                  len(repetidas), int(imdb_original.duplicated("review").sum()),
                  int((repetidas.groupby("review")["sentiment"].nunique() > 1).sum()),
                  f"{imdb_original['review'].str.contains('<br', regex=False).mean() * 100:.1f} %"],
})
display(validacion)

imdb = imdb_original.drop_duplicates("review").reset_index(drop=True)
imdb["y"] = (imdb["sentiment"] == "positive").astype(np.int64)
print(f"Reseñas después de eliminar duplicados: {len(imdb):,}")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Número de duplicados y si alguno tiene etiquetas contradictorias.
- Proporción de reseñas con etiquetas HTML (justifica la limpieza de la sección 3.4).

### 3.3 Distribución de clases

Se cuenta cuántas reseñas tiene cada clase después de eliminar duplicados. Si hubiera desbalance, la exactitud podría ser alta solo por acertar la clase frecuente y habría que probar una corrección (pesos de clase).

In [ ]:
# ============================================================
# Reseñas por sentimiento
# ============================================================
tabla_clases = imdb["y"].value_counts().sort_index().rename(index=dict(enumerate(SENTIMIENTOS))).to_frame("reseñas")
tabla_clases["%"] = tabla_clases["reseñas"] / tabla_clases["reseñas"].sum() * 100
display(tabla_clases.round(2))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Proporción de cada clase después de eliminar duplicados.
- Si las clases quedan balanceadas, indicar que no se necesita corrección del desbalance.

### 3.4 Limpieza y tokenización

Cada reseña se transforma en una lista de tokens en tres pasos:

1. **Minúsculas:** «Great» y «great» son la misma palabra; sin este paso el vocabulario se duplica.
2. **Eliminación de `<br />`:** las etiquetas HTML no aportan sentimiento y aparecerían como tokens muy frecuentes.
3. **Separación en tokens:** palabras (letras, dígitos y apóstrofos, de modo que «don't» queda como un solo token) y los signos `! ? . , ; :` como tokens propios. Se conservan los signos porque expresan intensidad («*great!!!*») y separan cláusulas («*boring, but*»).

`tokenizar(texto)` retorna la lista de tokens de una reseña.

In [ ]:
# ============================================================
# Limpieza y tokenización
# ============================================================
PATRON_HTML = re.compile(r"<br\s*/?>")
PATRON_TOKENS = re.compile(r"[a-z0-9']+|[!?.,;:]")


def tokenizar(texto):
    return PATRON_TOKENS.findall(PATRON_HTML.sub(" ", texto.lower()))


imdb["tokens"] = imdb["review"].map(tokenizar)
imdb["largo"] = imdb["tokens"].str.len()
ejemplo = imdb.loc[imdb["review"].str.contains("<br", regex=False)].iloc[0]
print("Original :", ejemplo["review"][:300])
print("Tokens   :", ejemplo["tokens"][:60])

### 3.5 Largo de las reseñas y largo máximo

Las redes procesan lotes de secuencias, así que se fija un **largo máximo** (`maxlen`): las reseñas más largas se truncan. Un `maxlen` pequeño descarta parte del texto; uno grande hace cada época más lenta, sobre todo en el Transformer, cuyo costo de atención crece con el **cuadrado** del largo.

Se muestran la distribución del largo por sentimiento y, para varios `maxlen`, el porcentaje de reseñas truncadas y de tokens conservados. Se fija `maxlen = 320`, que conserva cerca del **80 % de los tokens** del dataset. Se trunca conservando el **inicio** de la reseña; la sección 3.8 mide con la regla léxica si el inicio o el final contienen más señal.

In [ ]:
# ============================================================
# Largo de las reseñas por sentimiento
# ============================================================
fig = go.Figure([go.Histogram(x=imdb.loc[imdb["y"] == c, "largo"], name=SENTIMIENTOS[c], nbinsx=120, opacity=0.7,
                              marker_color=COLORES[2 * c]) for c in [0, 1]])
fig.add_vline(x=320, line_dash="dash", line_color=AMARILLO, annotation_text="maxlen = 320")
fig.update_layout(title="Largo de las reseñas (tokens)", barmode="overlay", xaxis_title="tokens",
                  yaxis_title="reseñas", xaxis_range=[0, 1500], height=420)
fig.show()

In [ ]:
# ============================================================
# Cobertura según el largo máximo
# ============================================================
MAXLEN = 320
largos = imdb["largo"].to_numpy()
tabla_maxlen = pd.DataFrame([{"maxlen": m, "reseñas_truncadas_%": (largos > m).mean() * 100,
                              "tokens_conservados_%": np.minimum(largos, m).sum() / largos.sum() * 100}
                             for m in [100, 128, 200, 256, 300, 320, 400, 512]])
print(imdb.groupby("y")["largo"].describe().rename(index=dict(enumerate(SENTIMIENTOS))).round(1))
display(tabla_maxlen.round(2))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mediana y percentiles del largo, y si difieren entre reseñas positivas y negativas.
- Porcentaje de tokens conservados y de reseñas truncadas con maxlen = 320.

### 3.6 Ejemplos por clase

Se muestran dos reseñas cortas de cada clase, elegidas al azar con la semilla fija, para reconocer el tipo de señal que debe aprender el modelo (adjetivos, negaciones, contrastes).

In [ ]:
# ============================================================
# Ejemplos de reseñas cortas por clase
# ============================================================
cortas = imdb[imdb["largo"] < 60]
ejemplos = pd.concat([cortas[cortas["y"] == c].sample(2, random_state=SEED) for c in [0, 1]])
display(ejemplos.assign(sentimiento=ejemplos["y"].map(dict(enumerate(SENTIMIENTOS))))[["sentimiento", "largo", "review"]])

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Palabras o construcciones que delatan el sentimiento.
- Casos en que el sentimiento depende del orden o de una negación.

### 3.7 Partición en entrenamiento, validación y test

Las 49.582 reseñas sin duplicados se dividen en **34.708 / 7.437 / 7.437** (70 / 15 / 15 %), estratificando por sentimiento y con la semilla fija. Son casi las mismas proporciones de la partición de Fashion-MNIST (50.000 / 10.000 / 10.000), y validación y test tienen el mismo tamaño en ambos datasets: en el notebook de fusión, cada imagen se empareja con una reseña de la misma partición, y así cada reseña se repite un número parecido de veces en las tres. Los índices se guardan en el respaldo para que la fusión use exactamente esta partición.

In [ ]:
# ============================================================
# Partición estratificada 34.708 / 7.437 / 7.437
# ============================================================
N_VAL = N_TEST = 7437
idx_train, idx_resto = train_test_split(np.arange(len(imdb)), test_size=N_VAL + N_TEST, stratify=imdb["y"],
                                        random_state=SEED)
idx_val, idx_test = train_test_split(idx_resto, test_size=N_TEST, stratify=imdb["y"].to_numpy()[idx_resto],
                                     random_state=SEED)
particiones = {"train": idx_train, "val": idx_val, "test": idx_test}
display(pd.DataFrame({nombre: {"reseñas": len(idx), "% positivas": imdb["y"].to_numpy()[idx].mean() * 100}
                      for nombre, idx in particiones.items()}).T.round(2))

### 3.8 Líneas base

Fijan el piso que los modelos deben superar:

- **Clase mayoritaria:** predice siempre la clase más frecuente del entrenamiento (≈ 50 % de exactitud con clases balanceadas).
- **Regla léxica sin aprendizaje:** cuenta 10 palabras claramente positivas y 10 claramente negativas, elegidas a mano antes de mirar los datos; predice positiva si hay al menos tantas positivas como negativas. Mide cuánto se resuelve solo con palabras clave. Se aplica además a los primeros y a los últimos 320 tokens, para ver dónde está la señal y justificar el lado del truncado.
- **Referencia lineal (TF-IDF + regresión logística):** pondera cada palabra por su frecuencia en la reseña y su rareza en el corpus, y aprende un peso por palabra. Ignora por completo el orden. Si los modelos secuenciales no la superan, el orden de las palabras no está aportando.

`metricas(y_true, y_pred)` retorna un diccionario con exactitud, precisión macro, exhaustividad macro, F1 macro y F1 ponderado. `predecir_regla(listas_tokens)` retorna la predicción de la regla léxica.

In [ ]:
# ============================================================
# Métricas y líneas base
# ============================================================
PALABRAS_POSITIVAS = {"great", "excellent", "best", "love", "wonderful", "perfect", "amazing", "beautiful", "favorite", "brilliant"}
PALABRAS_NEGATIVAS = {"bad", "worst", "awful", "waste", "terrible", "boring", "poor", "horrible", "stupid", "worse"}


def metricas(y_true, y_pred):
    precision, exhaustividad, _, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"exactitud": accuracy_score(y_true, y_pred), "precision_macro": precision,
            "exhaustividad_macro": exhaustividad, "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "f1_ponderado": f1_score(y_true, y_pred, average="weighted")}


def predecir_regla(listas_tokens):
    return np.array([int(sum(t in PALABRAS_POSITIVAS for t in toks) >= sum(t in PALABRAS_NEGATIVAS for t in toks))
                     for toks in listas_tokens])


tokens_train, tokens_val = imdb["tokens"].to_numpy()[idx_train], imdb["tokens"].to_numpy()[idx_val]
y_train, y_val, y_test = (imdb["y"].to_numpy()[i] for i in (idx_train, idx_val, idx_test))
CLASE_MAYORITARIA = int(np.bincount(y_train).argmax())
tfidf = TfidfVectorizer(analyzer=lambda toks: toks, min_df=5, sublinear_tf=True)
regresion = LogisticRegression(max_iter=2000).fit(tfidf.fit_transform(tokens_train), y_train)
tabla_lineas_base_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(y_val, np.full(len(y_val), CLASE_MAYORITARIA))},
    {"modelo": "Regla léxica (reseña completa)", **metricas(y_val, predecir_regla(tokens_val))},
    {"modelo": "Regla léxica (primeros 320 tokens)", **metricas(y_val, predecir_regla([t[:MAXLEN] for t in tokens_val]))},
    {"modelo": "Regla léxica (últimos 320 tokens)", **metricas(y_val, predecir_regla([t[-MAXLEN:] for t in tokens_val]))},
    {"modelo": "TF-IDF + regresión logística", **metricas(y_val, regresion.predict(tfidf.transform(tokens_val)))},
])
display(tabla_lineas_base_val.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de cada línea base en validación: el piso de los modelos secuenciales.
- Diferencia entre la regla aplicada al inicio y al final de la reseña: ¿justifica truncar conservando el inicio?
- F1 de la referencia lineal: la cifra que un modelo secuencial debería superar para justificar su costo.

## 4. Preprocesamiento

### 4.1 Vocabulario

`Vocab` asigna un índice a cada token que aparece al menos `min_freq` veces en el **entrenamiento** y envía el resto a `<unk>` (palabra desconocida). Se construye solo con entrenamiento para no filtrar información de validación ni de test. Es una clase mínima propia en vez de `torchtext`, librería descontinuada que ya no es compatible con las versiones actuales de PyTorch.

- Tokens especiales: `<unk>`, `<pad>` (relleno, para igualar el largo dentro de un lote) y `<cls>` (token de resumen que usa una de las variantes del Transformer, sección 16).
- `min_freq` controla el tamaño del vocabulario y, con él, los parámetros del *embedding* (tamaño del vocabulario × 128). La tabla muestra, para varios umbrales, el tamaño y el porcentaje de tokens de validación que quedarían como `<unk>`. Se fija `min_freq = 5`: descarta las palabras que aparecen menos de 5 veces (en su mayoría nombres propios y errores de tipeo), con las que el modelo no alcanzaría a aprender un vector útil.

In [ ]:
# ============================================================
# Vocabulario
# ============================================================
class Vocab:
    def __init__(self, counter, min_freq=1, specials=("<unk>",)):
        self.itos = list(specials) + sorted(t for t, c in counter.items() if c >= min_freq and t not in specials)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.default_index = self.stoi["<unk>"]

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __len__(self):
        return len(self.itos)

    def get_itos(self):
        return self.itos


ESPECIALES = ("<unk>", "<pad>", "<cls>")
MIN_FREQ = 5
contador_train = Counter(t for toks in tokens_train for t in toks)
contador_val = Counter(t for toks in tokens_val for t in toks)
total_val = sum(contador_val.values())
tabla_vocab = pd.DataFrame([{"min_freq": m, "vocabulario": len(Vocab(contador_train, m, ESPECIALES)),
                             "tokens_val_desconocidos_%": sum(c for t, c in contador_val.items()
                                                              if contador_train.get(t, 0) < m) / total_val * 100}
                            for m in [1, 2, 3, 5, 10, 20]])
display(tabla_vocab.round(2))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Tamaño del vocabulario y porcentaje de tokens desconocidos en validación con min_freq = 5.
- Parámetros del embedding que se ahorran frente a min_freq = 1.

### 4.2 Codificación y tensores en GPU

Cada reseña se convierte en sus índices del vocabulario, truncada a los primeros `MAXLEN` tokens y rellenada con `<pad>` hasta `MAXLEN`. Las tres particiones se copian una sola vez a la GPU; cada lote se recorta luego al largo de su reseña más larga, para no procesar relleno innecesario.

`codificar(indices)` retorna un diccionario con la matriz de índices `x` (`[reseñas, MAXLEN]`, en GPU), el largo real de cada reseña (en GPU y en CPU), la etiqueta `y` y el largo original antes de truncar (para analizar el desempeño por largo).

In [ ]:
# ============================================================
# Codificación de las reseñas y tensores en GPU
# ============================================================
VOCAB = Vocab(contador_train, MIN_FREQ, ESPECIALES)
N_VOCAB, PAD_IDX, CLS_IDX, UNK_IDX = len(VOCAB), VOCAB["<pad>"], VOCAB["<cls>"], VOCAB["<unk>"]


def codificar(indices):
    x = torch.full((len(indices), MAXLEN), PAD_IDX, dtype=torch.long)
    largos = torch.zeros(len(indices), dtype=torch.long)
    for fila, toks in enumerate(imdb["tokens"].to_numpy()[indices]):
        ids = [VOCAB[t] for t in toks[:MAXLEN]] or [UNK_IDX]
        x[fila, :len(ids)] = torch.tensor(ids)
        largos[fila] = len(ids)
    return {"x": x.to(device), "largos": largos.to(device), "largos_cpu": largos,
            "y": torch.tensor(imdb["y"].to_numpy()[indices], device=device),
            "largo_original": imdb["largo"].to_numpy()[indices]}


DATOS_TRAIN, DATOS_VAL, DATOS_TEST = codificar(idx_train), codificar(idx_val), codificar(idx_test)
print(f"Vocabulario: {N_VOCAB:,} tokens | tensor de entrenamiento: {tuple(DATOS_TRAIN['x'].shape)}")
print("Reconstrucción de la primera reseña:", " ".join(VOCAB.get_itos()[i] for i in DATOS_TRAIN["x"][0, :40].tolist()))

### 4.3 Lotes por largo

Con largos que van de menos de 10 a 320 tokens, un lote armado al azar mezcla reseñas cortas con largas y la mayor parte del cómputo se gasta en relleno. `lotes_por_largo(largos, batch_size, generador)` baraja las reseñas, las agrupa en bloques de 50 lotes, ordena cada bloque por largo y lo corta en lotes, de modo que cada lote reúne reseñas de largo parecido; finalmente baraja el orden de los lotes. Retorna una lista de tensores de índices, uno por lote. Así se conserva el azar entre épocas y el relleno se reduce.

In [ ]:
# ============================================================
# Lotes de reseñas de largo parecido
# ============================================================
def lotes_por_largo(largos, batch_size, generador):
    orden = torch.randperm(len(largos), generator=generador)
    lotes = []
    for inicio in range(0, len(orden), batch_size * 50):
        bloque = orden[inicio:inicio + batch_size * 50]
        bloque = bloque[torch.argsort(largos[bloque], descending=True)]
        lotes += list(torch.split(bloque, batch_size))
    return [lotes[i] for i in torch.randperm(len(lotes), generator=generador).tolist()]


lotes_ejemplo = lotes_por_largo(DATOS_TRAIN["largos_cpu"], 64, torch.Generator().manual_seed(SEED))
relleno_ordenado = np.mean([1 - DATOS_TRAIN["largos_cpu"][l].float().mean().item() / DATOS_TRAIN["largos_cpu"][l].max().item()
                            for l in lotes_ejemplo])
lotes_azar = torch.split(torch.randperm(len(DATOS_TRAIN["largos_cpu"]), generator=torch.Generator().manual_seed(SEED)), 64)
relleno_azar = np.mean([1 - DATOS_TRAIN["largos_cpu"][l].float().mean().item() / DATOS_TRAIN["largos_cpu"][l].max().item()
                        for l in lotes_azar])
print(f"Lotes por época con batch 64: {len(lotes_ejemplo)} | relleno medio por lote: {relleno_ordenado * 100:.1f} % "
      f"(lotes al azar: {relleno_azar * 100:.1f} %)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Porcentaje de relleno con lotes por largo frente a lotes al azar: cómputo que se ahorra.

## 5. Utilidades de entrenamiento, evaluación y experimentos

### 5.1 Una época de entrenamiento y predicción

- `entrenar_epoca(modelo, datos, config, optimizador, criterio, generador)`: recorre los lotes por largo, calcula la pérdida (entropía cruzada), hace *backpropagation*, recorta la norma del gradiente a `clip` (`clip_grad_norm_`, que evita los saltos bruscos típicos de las redes recurrentes) y actualiza los pesos. Retorna la pérdida y la exactitud de entrenamiento.
- `predecir(modelo, datos, criterio)`: evalúa sin gradientes, en lotes ordenados por largo, y devuelve los resultados en el orden original de las reseñas. Retorna la pérdida (si se entrega un criterio), las etiquetas reales, las predichas y las probabilidades.

In [ ]:
# ============================================================
# Entrenamiento de una época y predicción
# ============================================================
def entrenar_epoca(modelo, datos, config, optimizador, criterio, generador):
    modelo.train()
    perdida, aciertos, total = 0.0, 0, 0
    for indices in lotes_por_largo(datos["largos_cpu"], config["batch_size"], generador):
        indices = indices.to(device)
        largos = datos["largos"][indices]
        x, y = datos["x"][indices, :int(largos.max())], datos["y"][indices]
        optimizador.zero_grad()
        logits = modelo(x, largos)
        loss = criterio(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), config["clip"])
        optimizador.step()
        perdida += loss.item() * len(y)
        aciertos += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return perdida / total, aciertos / total


def predecir(modelo, datos, criterio=None, lote=512):
    modelo.eval()
    orden = torch.argsort(datos["largos_cpu"], descending=True).to(device)
    logits_todos = torch.zeros(len(orden), 2, device=device)
    perdida = 0.0
    with torch.no_grad():
        for inicio in range(0, len(orden), lote):
            indices = orden[inicio:inicio + lote]
            largos = datos["largos"][indices]
            logits = modelo(datos["x"][indices, :int(largos.max())], largos)
            if criterio is not None:
                perdida += criterio(logits, datos["y"][indices]).item() * len(indices)
            logits_todos[indices] = logits
    return {"perdida": perdida / len(orden), "y_true": datos["y"].cpu().numpy(),
            "y_pred": logits_todos.argmax(1).cpu().numpy(), "probas": torch.softmax(logits_todos, 1).cpu().numpy()}

### 5.2 Entrenamiento con Early Stopping y registro de métricas

`entrenar_y_registrar(config, semilla, epocas, paciencia, verbose, forzar)` construye el modelo de la familia indicada en `config["familia"]` y lo entrena con **Early Stopping**: monitorea la pérdida de validación y, al terminar, restaura los pesos de la mejor época. Con `paciencia=0` entrena todas las épocas sin detenerse. Retorna `(fila, historia, modelo)`: la fila de métricas en la mejor época, la historia por época y el modelo entrenado.

- **Régimen uniforme para ambas familias:** `EPOCAS_EXPERIMENTO = 15` y `PACIENCIA_EXPERIMENTO = 3`, incluidos los modelos finales. Las reseñas de IMDB se sobreajustan pronto (unas pocas épocas bastan para memorizar el entrenamiento), así que un tope de 15 deja margen holgado; los experimentos de épocas sin Early Stopping (secciones 10.3 y 17.3) muestran dónde empieza el sobreajuste.
- **Extensión sin reiniciar:** si la mejor época queda en las dos últimas del tope, la corrida **continúa** hasta `EPOCAS_EXTENDIDAS = 25` en lugar de empezar de nuevo.
- Ambas familias se optimizan con **Adam** (Kingma y Ba, 2015), que ajusta la tasa por parámetro.
- Si existe un resultado guardado con la misma clave y `forzar=False`, se reutiliza y no se entrena; en ese caso no hay modelo ni historia.
- Cada corrida nueva se guarda en el JSON y se imprime como `CORRIDA_GUARDADA {...}`.
- Si la pérdida deja de ser finita (el entrenamiento **diverge**), la corrida se detiene, se marca con `diverge = True` y se registra como **Error / Falla** en vez de interrumpir el notebook.

In [ ]:
# ============================================================
# Entrenamiento con Early Stopping, extensión y registro de métricas
# ============================================================
EPOCAS_EXPERIMENTO = 15
PACIENCIA_EXPERIMENTO = 3
EPOCAS_EXTENDIDAS = 25
CRITERIO = nn.CrossEntropyLoss()


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


def clave_corrida(config, semilla, epocas, paciencia):
    return json.dumps({**config, "semilla": semilla, "epocas": epocas, "paciencia": paciencia}, sort_keys=True)


def entrenar_y_registrar(config, semilla=SEED, epocas=None, paciencia=None, verbose=False, forzar=False):
    epocas = epocas or EPOCAS_EXPERIMENTO
    paciencia = PACIENCIA_EXPERIMENTO if paciencia is None else paciencia
    clave = clave_corrida(config, semilla, epocas, paciencia)
    if not forzar and clave in RESULTADOS_GUARDADOS:
        return {**RESULTADOS_GUARDADOS[clave], "origen": "guardado"}, None, None

    fijar_semilla(semilla)
    generador = torch.Generator().manual_seed(semilla)
    modelo = construir_modelo(config, N_VOCAB, PAD_IDX, CLS_IDX)
    if forzar and REUTILIZAR_MODELOS:
        guardada = cargar_modelo_corrida(clave, modelo)
        if guardada is not None:
            fila, historia = guardada
            print(f"Corrida cargada desde {nombre_modelo_corrida(clave)} (sin entrenar)")
            return {**fila, "origen": "modelo guardado"}, historia, modelo
    optimizador = torch.optim.Adam(modelo.parameters(), lr=config["learning_rate"])
    historia = {"loss_train": [], "exactitud_train": [], "loss_val": [], "exactitud_val": [], "f1_val": []}
    mejor_loss, mejor_estado, mejor_epoca, sin_mejora, diverge = float("inf"), None, 0, 0, False
    tope, epoca, inicio = epocas, 0, time.time()
    while epoca < tope:
        loss_tr, acc_tr = entrenar_epoca(modelo, DATOS_TRAIN, config, optimizador, CRITERIO, generador)
        val = predecir(modelo, DATOS_VAL, CRITERIO)
        acc_val = accuracy_score(val["y_true"], val["y_pred"])
        f1_val = f1_score(val["y_true"], val["y_pred"], average="macro")
        for k, valor in zip(historia, [loss_tr, acc_tr, val["perdida"], acc_val, f1_val]):
            historia[k].append(valor)
        if verbose:
            print(f"Época {epoca + 1:02} | loss train {loss_tr:.4f} | loss val {val['perdida']:.4f} "
                  f"| exactitud val {acc_val:.4f} | F1 macro val {f1_val:.4f}")
        if not (np.isfinite(loss_tr) and np.isfinite(val["perdida"])):
            print(f"  Error / Falla: la pérdida dejó de ser finita en la época {epoca + 1} (el entrenamiento divergió).")
            diverge = True
            break
        if val["perdida"] < mejor_loss:
            mejor_loss, mejor_epoca, sin_mejora = val["perdida"], epoca, 0
            mejor_estado = copy.deepcopy(modelo.state_dict())
        else:
            sin_mejora += 1
            if paciencia and sin_mejora >= paciencia:
                break
        epoca += 1
        if epoca == tope and paciencia and mejor_epoca >= tope - 2 and tope < EPOCAS_EXTENDIDAS:
            print(f"  La mejor época ({mejor_epoca + 1}) está en el tope: se continúa hasta {EPOCAS_EXTENDIDAS} épocas.")
            tope = EPOCAS_EXTENDIDAS
    duracion = time.time() - inicio
    if mejor_estado is None:
        mejor_estado = copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(mejor_estado)
    if paciencia and not diverge and mejor_epoca >= tope - 2:
        print(f"  Aviso: la mejor época ({mejor_epoca + 1}) sigue al final del tope extendido: la variante no convergió.")

    m = mejor_epoca
    fila = {
        "exactitud_train": historia["exactitud_train"][m],
        "exactitud_val": historia["exactitud_val"][m],
        "f1_macro_val": historia["f1_val"][m],
        "loss_val": historia["loss_val"][m],
        "brecha_train_val": historia["exactitud_train"][m] - historia["exactitud_val"][m],
        "mejor_epoca": m + 1,
        "epocas_entrenadas": len(historia["loss_train"]),
        "parametros": contar_parametros(modelo),
        "segundos": duracion,
        "segundos_por_epoca": duracion / len(historia["loss_train"]),
        "diverge": diverge,
    }
    CORRIDAS_NUEVAS[clave] = fila
    if forzar:
        guardar_modelo_corrida(clave, modelo, historia, fila)
    print("CORRIDA_GUARDADA " + json.dumps({"clave": clave, "fila": fila}, default=float))
    guardar_corridas()
    return {**fila, "origen": "entrenado"}, historia, modelo

### 5.3 Experimentos controlados, selección y ampliación del rango

Cada familia lleva su propia **cadena de decisiones**: un diccionario con la configuración vigente, su rango de error, los experimentos hechos y las decisiones tomadas. Así, las decisiones de la red recurrente no afectan al Transformer.

- `nueva_cadena(nombre, config_base, margen)`: crea la cadena de una familia con su configuración base y su `MARGEN_ERROR`.
- `ejecutar_experimento(cadena, nombre, variantes, forzar)`: entrena (o recupera) las **variantes** `{etiqueta: cambios}`, que modifican un solo hiperparámetro respecto de la configuración vigente. Retorna una tabla ordenada por **F1 macro de validación**, con la diferencia respecto de la mejor y si cae dentro del rango de error. Las historias y los modelos de las variantes entrenadas quedan en `historias_ultimo_experimento` y `modelos_ultimo_experimento`. Si se repite un experimento, la configuración vuelve a su estado previo y se descartan las decisiones posteriores.
- `ampliar_rango(cadena, nombre, variantes, hiperparametro, arriba, abajo, minimo, maximo)`: si la mejor variante queda en el **borde** del rango probado, agrega el valor siguiente en esa dirección y lo entrena, mientras el ganador siga en el borde y no se supere el límite. Retorna la tabla actualizada.
- `seleccionar_variante(cadena, tabla)`: **propone** la mejor variante y lista las que están en empate técnico. No decide los empates: esos se revisan caso a caso y la decisión queda en una línea `# Cambio Manual`.
- `aplicar_variante(cadena, nombre, variantes, elegida)`: incorpora la variante a la configuración de la cadena, registra la decisión con su F1 y el de la configuración previa, y muestra la configuración.
- `tabla_de(cadena, nombre)`: retorna la tabla guardada de un experimento.

In [ ]:
# ============================================================
# Experimentos controlados (un hiperparámetro a la vez, por familia)
# ============================================================
historias_ultimo_experimento = {}
modelos_ultimo_experimento = {}
COLUMNAS_TABLA = ["variante", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca",
                  "epocas_entrenadas", "parametros", "segundos_por_epoca", "diferencia_vs_mejor", "dentro_del_margen",
                  "diverge", "origen"]


def nueva_cadena(nombre, config_base, margen):
    return {"nombre": nombre, "config": dict(config_base), "margen": margen, "historial": [], "resultados": [],
            "decisiones": {}}


def entrenar_variantes(cadena, nombre, variantes, forzar=False):
    filas = []
    for etiqueta, cambios in variantes.items():
        fila, historia, modelo = entrenar_y_registrar({**cadena["config"], **cambios}, forzar=forzar)
        print(f"[{cadena['nombre']} · {nombre}] {etiqueta}: {fila['origen']}")
        if historia is not None:
            historias_ultimo_experimento[etiqueta] = historia
            modelos_ultimo_experimento[etiqueta] = modelo
        filas.append({"experimento": nombre, "variante": etiqueta, **fila})
    return filas


def armar_tabla(filas, margen):
    tabla = pd.DataFrame(filas)
    tabla["diferencia_vs_mejor"] = tabla["f1_macro_val"].max() - tabla["f1_macro_val"]
    tabla["dentro_del_margen"] = tabla["diferencia_vs_mejor"] <= margen
    return tabla.sort_values("f1_macro_val", ascending=False).reset_index(drop=True)


def ejecutar_experimento(cadena, nombre, variantes, forzar=False):
    nombres = [n for n, _ in cadena["historial"]]
    if nombre in nombres:
        i = nombres.index(nombre)
        cadena["config"] = dict(cadena["historial"][i][1])
        for n in nombres[i:]:
            cadena["decisiones"].pop(n, None)
        del cadena["historial"][i:]
        del cadena["resultados"][i:]
        print(f"[{nombre}] Experimento repetido: la configuración vuelve a su estado previo "
              f"y se descartan las decisiones posteriores ({', '.join(nombres[i:])}).")
    cadena["historial"].append((nombre, dict(cadena["config"])))
    historias_ultimo_experimento.clear()
    modelos_ultimo_experimento.clear()
    tabla = armar_tabla(entrenar_variantes(cadena, nombre, variantes, forzar), cadena["margen"])
    cadena["resultados"].append(tabla)
    return tabla


def ampliar_rango(cadena, nombre, variantes, hiperparametro, arriba, abajo, minimo, maximo):
    tabla = cadena["resultados"][-1]
    while True:
        valores = {etq: cambios[hiperparametro] for etq, cambios in variantes.items()}
        mejor = valores[tabla.loc[0, "variante"]]
        if mejor == max(valores.values()) and arriba(mejor) <= maximo:
            nuevo = arriba(mejor)
        elif mejor == min(valores.values()) and abajo(mejor) >= minimo:
            nuevo = abajo(mejor)
        else:
            return tabla
        etiqueta = f"{nuevo:g}" if isinstance(nuevo, float) else str(nuevo)
        print(f"[{nombre}] La mejor variante está en el borde del rango: se agrega {hiperparametro} = {etiqueta}")
        variantes[etiqueta] = {hiperparametro: nuevo}
        filas = tabla.drop(columns=["diferencia_vs_mejor", "dentro_del_margen"]).to_dict("records")
        filas += entrenar_variantes(cadena, nombre, {etiqueta: {hiperparametro: nuevo}})
        tabla = armar_tabla(filas, cadena["margen"])
        cadena["resultados"][-1] = tabla


def seleccionar_variante(cadena, tabla):
    mejor = tabla.loc[0, "variante"]
    empatadas = tabla.loc[tabla["dentro_del_margen"] & (tabla["variante"] != mejor), "variante"].tolist()
    print(f"Mejor F1 macro de validación: {mejor}")
    if empatadas:
        print(f"En empate técnico (dentro de {cadena['margen'] * 100:.2f} pp): {', '.join(map(str, empatadas))}")
    return mejor


def tabla_de(cadena, nombre):
    return cadena["resultados"][[n for n, _ in cadena["historial"]].index(nombre)]


def aplicar_variante(cadena, nombre, variantes, elegida):
    tabla, config_previa = cadena["resultados"][-1], cadena["historial"][-1][1]
    previa = [etq for etq, cambios in variantes.items() if all(config_previa.get(k) == v for k, v in cambios.items())]
    f1_de = dict(zip(tabla["variante"], tabla["f1_macro_val"]))
    cadena["decisiones"][nombre] = {"elegida": elegida, "f1_elegida": f1_de[elegida],
                                    "variante_previa": previa[0] if previa else None,
                                    "f1_previa": f1_de[previa[0]] if previa else None}
    cadena["config"].update(variantes[elegida])
    mostrar_config(cadena["config"])

### 5.4 Curvas de entrenamiento, configuración y desempeño por largo

- `graficar_curvas(historia, titulo)`: pérdida, exactitud y F1 macro de validación por época en tres paneles horizontales, con la mejor época marcada.
- `graficar_comparacion(historias, clave, titulo, eje_y)`: una línea por variante para la métrica `clave`.
- `mostrar_config(config)`: muestra la configuración como tabla vertical.
- `f1_por_largo(y_true, y_pred, largo_original)`: divide las reseñas en tres tramos según su largo original (hasta 128 tokens, de 129 a 320 y más de 320, es decir, truncadas) y retorna una tabla con el número de reseñas, la exactitud y el F1 macro de cada tramo. Sirve para ver si un modelo pierde la señal en las reseñas largas.

In [ ]:
# ============================================================
# Visualización de curvas, configuración y desempeño por largo
# ============================================================
TRAMOS = [(0, 128, "≤ 128"), (129, 320, "129–320"), (321, 10 ** 6, "> 320 (truncada)")]


def graficar_curvas(historia, titulo):
    epocas = list(range(1, len(historia["loss_train"]) + 1))
    mejor = int(np.argmin(historia["loss_val"])) + 1
    fig = make_subplots(rows=1, cols=3, subplot_titles=["Pérdida", "Exactitud", "F1 macro (validación)"])
    for col, tr, va in [(1, "loss_train", "loss_val"), (2, "exactitud_train", "exactitud_val")]:
        fig.add_trace(go.Scatter(x=epocas, y=historia[tr], name="entrenamiento", line_color=COLORES[2],
                                 legendgroup="train", showlegend=col == 1), row=1, col=col)
        fig.add_trace(go.Scatter(x=epocas, y=historia[va], name="validación", line_color=COLORES[0],
                                 legendgroup="val", showlegend=col == 1), row=1, col=col)
    fig.add_trace(go.Scatter(x=epocas, y=historia["f1_val"], name="F1 macro validación", line_color=COLORES[1]), row=1, col=3)
    for col in [1, 2, 3]:
        fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
        fig.update_xaxes(title_text="época", row=1, col=col)
    fig.update_layout(title=f"{titulo} · mejor época: {mejor}", height=420)
    fig.show()


def graficar_comparacion(historias, clave, titulo, eje_y):
    paleta = COLORES + ["#0E2747", "#8A6A00", "#52514E"]
    fig = go.Figure()
    for i, (etiqueta, historia) in enumerate(historias.items()):
        fig.add_trace(go.Scatter(x=list(range(1, len(historia[clave]) + 1)), y=historia[clave], name=etiqueta,
                                 line_color=paleta[i % len(paleta)]))
    fig.update_layout(title=titulo, xaxis_title="época", yaxis_title=eje_y, height=420)
    fig.show()


def mostrar_config(config):
    display(pd.DataFrame({"valor": pd.Series(config).astype(str)}))


def f1_por_largo(y_true, y_pred, largo_original):
    filas = []
    for minimo, maximo, etiqueta in TRAMOS:
        en_tramo = (largo_original >= minimo) & (largo_original <= maximo)
        filas.append({"tramo": etiqueta, "reseñas": int(en_tramo.sum()),
                      "exactitud": accuracy_score(y_true[en_tramo], y_pred[en_tramo]),
                      "f1_macro": f1_score(y_true[en_tramo], y_pred[en_tramo], average="macro")})
    return pd.DataFrame(filas)

## 6. Arquitecturas

### 6.1 Bloques del Transformer: autoatención, encoder y decoder

El Transformer (Vaswani et al., 2017) reemplaza la recurrencia por **atención**. Los bloques que se definen en la celda siguiente son:

- **`MultiHeadAttentionLayer` (atención de múltiples cabezas):** cada token genera una consulta (Q), una clave (K) y un valor (V). La similitud entre la consulta de un token y las claves de todos los demás, escalada por √(dimensión de la cabeza) y normalizada con *softmax*, da los **pesos de atención**; la salida es el promedio de los valores ponderado por esos pesos. Con varias **cabezas**, cada una aprende un tipo de relación distinto (por ejemplo, una negación con el adjetivo que niega). La **máscara** asigna peso cero a los tokens de relleno.
- **Autoatención:** Q, K y V salen de la misma secuencia, de modo que cada token se relaciona directamente con todos los demás, a cualquier distancia, en un solo paso. En una red recurrente, la información del token 1 debe atravesar 300 actualizaciones del estado para llegar al token 300.
- **`PositionwiseFeedforwardLayer`:** una red densa de dos capas (`hid_dim` → `pf_dim` → `hid_dim`) que se aplica a cada token por separado.
- **`EncoderBlock`:** autoatención + *feed-forward*, cada uno con conexión residual y normalización de capa (*LayerNorm*). Varios bloques apilados forman el **encoder**.
- **`DecoderLayer`:** autoatención sobre su propia entrada, **atención cruzada** (las consultas vienen del decoder y las claves y valores, de la salida del encoder) y *feed-forward*.
- **Posición:** la atención no distingue el orden por sí sola; se suma a cada token un *embedding* de su posición, aprendido, con `maxlen + 1` posiciones.

In [ ]:
# ============================================================
# Atención de múltiples cabezas, bloque encoder y bloque decoder
# ============================================================
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout):
        super().__init__()
        assert hid_dim % n_heads == 0
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads
        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)
        self.fc_o = nn.Linear(hid_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        # Q = [batch size, query len, hid dim] -> [batch size, n heads, query len, head dim]
        Q = self.fc_q(query).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.fc_k(key).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.fc_v(value).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        # energy = [batch size, n heads, query len, key len]
        if mask is not None:
            energy = energy.masked_fill(~mask, -1e10)
        attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attention), V)
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.hid_dim)
        # x = [batch size, query len, hid dim]
        return self.fc_o(x), attention


class PositionwiseFeedforwardLayer(nn.Module):
    def __init__(self, hid_dim, pf_dim, dropout):
        super().__init__()
        self.fc_1 = nn.Linear(hid_dim, pf_dim)
        self.fc_2 = nn.Linear(pf_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc_2(self.dropout(torch.relu(self.fc_1(x))))


class EncoderBlock(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.guardar_atencion, self.atencion = False, None

    def forward(self, src, src_mask):
        # src = [batch size, src len, hid dim]; src_mask = [batch size, 1, 1, src len]
        _src, atencion = self.self_attention(src, src, src, src_mask)
        if self.guardar_atencion:
            self.atencion = atencion.detach()
        src = self.self_attn_layer_norm(src + self.dropout(_src))
        _src = self.positionwise_feedforward(src)
        return self.ff_layer_norm(src + self.dropout(_src))


class DecoderLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.enc_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.encoder_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.guardar_atencion, self.atencion = False, None

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # trg = [batch size, trg len, hid dim]; enc_src = [batch size, src len, hid dim]
        _trg, _ = self.self_attention(trg, trg, trg, trg_mask)
        trg = self.self_attn_layer_norm(trg + self.dropout(_trg))
        _trg, atencion = self.encoder_attention(trg, enc_src, enc_src, src_mask)
        if self.guardar_atencion:
            self.atencion = atencion.detach()
        trg = self.enc_attn_layer_norm(trg + self.dropout(_trg))
        _trg = self.positionwise_feedforward(trg)
        return self.ff_layer_norm(trg + self.dropout(_trg))

### 6.2 Clasificadores recurrente y Transformer

**`ClasificadorRecurrente`:** `Embedding` → capa(s) recurrente(s) → capa densa de representación (128) → salida (2)

- **`nn.Embedding`**: convierte cada índice en un vector denso de `embed_dim` valores que se aprende durante el entrenamiento. `padding_idx` deja fijo en cero el vector de `<pad>`.
- **Capa recurrente** (`nn.RNN`, `nn.LSTM` o `nn.GRU`): recorre la reseña token a token y actualiza su estado oculto hₜ.
  - **RNN simple:** hₜ = tanh(W·xₜ + U·hₜ₋₁ + b). Al retropropagar a través de cientos de pasos, el gradiente se multiplica una y otra vez por la misma matriz y tiende a desvanecerse: la red olvida el inicio de las reseñas largas.
  - **LSTM** (Hochreiter y Schmidhuber, 1997): agrega un estado de celda y compuertas de olvido, entrada y salida que deciden qué conservar; el estado de celda deja pasar el gradiente casi sin atenuarse.
  - **GRU** (Cho et al., 2014): simplifica la LSTM a dos compuertas (actualización y reinicio), con menos parámetros.
- **`pack_padded_sequence`**: le indica a la red el largo real de cada reseña, para que el estado final corresponda al último token real y no al relleno.
- Con `bidireccional=True` la reseña se lee en ambos sentidos y se concatenan los dos estados finales. Es válido porque se clasifica la reseña completa (no se genera texto).
- **Capa de representación:** una capa densa de 128 neuronas con ReLU, el mismo tamaño que la representación de la CNN. Es el vector que se entrega a la fusión, de modo que ambas modalidades aportan el mismo número de valores al concatenarse.
- La salida son 2 **logits**; `CrossEntropyLoss` aplica internamente el *softmax*.

`representacion(x, largos)` retorna el vector de 128 valores y `forward(x, largos)` los 2 logits.

**`ClasificadorTransformer`**: *embedding* de entrada de `embed_dim` (128), proyección a `hid_dim` si son distintos, *embedding* de posición, `capas` bloques encoder y una **forma de resumir** la reseña en un vector (`agregacion`), que se compara en la sección 16:

| `agregacion` | Cómo resume la reseña |
|---|---|
| `promedio` | Promedia los vectores de salida del encoder, sin contar el relleno |
| `cls` | Agrega el token `<cls>` al inicio; su vector de salida, que mediante la autoatención reúne información de toda la reseña, es el resumen |
| `decoder` | Un **decoder de un paso**: una consulta aprendida (un único vector de entrada del decoder) lee la salida del encoder mediante atención cruzada. Con una sola consulta, la autoatención interna del decoder solo se mira a sí misma; el trabajo lo hace la atención cruzada, cuyos pesos indican qué tokens usó el modelo para decidir |

Luego vienen la misma capa de representación de 128 neuronas y la salida de 2 logits de la red recurrente. `inicializar_pesos` aplica la inicialización de Xavier a las capas lineales y a los *embeddings*, que mantiene la varianza de las activaciones entre capas. `construir_modelo(config, n_vocab, pad_idx, cls_idx)` crea el modelo de la familia indicada en el dispositivo.

In [ ]:
# ============================================================
# Clasificador recurrente, clasificador Transformer y constructor
# ============================================================
CELDAS = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
UNIDADES_REPRESENTACION = 128


class ClasificadorRecurrente(nn.Module):
    def __init__(self, config, n_vocab, pad_idx):
        super().__init__()
        self.bidireccional = config["bidireccional"]
        self.embedding = nn.Embedding(n_vocab, config["embed_dim"], padding_idx=pad_idx)
        self.rnn = CELDAS[config["celda"]](config["embed_dim"], config["units"], num_layers=config["capas"],
                                           batch_first=True, bidirectional=config["bidireccional"],
                                           dropout=config["dropout"] if config["capas"] > 1 else 0.0)
        salida_rnn = config["units"] * (2 if self.bidireccional else 1)
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(salida_rnn, UNIDADES_REPRESENTACION), nn.ReLU())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(UNIDADES_REPRESENTACION, 2))

    def representacion(self, x, largos):
        # x = [batch size, seq len]
        empaquetado = pack_padded_sequence(self.embedding(x), largos.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(empaquetado)
        if isinstance(h, tuple):
            h = h[0]  # la LSTM devuelve (h, c)
        # h = [capas * direcciones, batch size, units]
        final = torch.cat([h[-2], h[-1]], dim=1) if self.bidireccional else h[-1]
        return self.densa(final)

    def forward(self, x, largos):
        return self.salida(self.representacion(x, largos))


class ClasificadorTransformer(nn.Module):
    def __init__(self, config, n_vocab, pad_idx, cls_idx):
        super().__init__()
        hid = config["hid_dim"]
        self.agregacion, self.pad_idx, self.cls_idx = config["agregacion"], pad_idx, cls_idx
        self.tok_embedding = nn.Embedding(n_vocab, config["embed_dim"])
        self.proyeccion = nn.Linear(config["embed_dim"], hid) if config["embed_dim"] != hid else nn.Identity()
        self.pos_embedding = nn.Embedding(config["maxlen"] + 1, hid)
        self.bloques = nn.ModuleList([EncoderBlock(hid, config["cabezas"], config["pf_dim"], config["dropout"])
                                      for _ in range(config["capas"])])
        if self.agregacion == "decoder":
            self.consulta = nn.Parameter(torch.randn(1, 1, hid) * 0.02)
            self.decoder = DecoderLayer(hid, config["cabezas"], config["pf_dim"], config["dropout"])
        self.dropout = nn.Dropout(config["dropout"])
        self.escala = math.sqrt(hid)
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(hid, UNIDADES_REPRESENTACION), nn.ReLU())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(UNIDADES_REPRESENTACION, 2))

    def codificar(self, x):
        if self.agregacion == "cls":
            x = torch.cat([torch.full((x.shape[0], 1), self.cls_idx, device=x.device), x], dim=1)
        mascara = (x != self.pad_idx)[:, None, None, :]
        # mascara = [batch size, 1, 1, src len]
        pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        h = self.dropout(self.proyeccion(self.tok_embedding(x)) * self.escala + self.pos_embedding(pos))
        for bloque in self.bloques:
            h = bloque(h, mascara)
        # h = [batch size, src len, hid dim]
        return h, mascara

    def representacion(self, x, largos):
        h, mascara = self.codificar(x)
        if self.agregacion == "promedio":
            validos = mascara[:, 0, 0, :].unsqueeze(-1).float()
            vector = (h * validos).sum(1) / validos.sum(1)
        elif self.agregacion == "cls":
            vector = h[:, 0]
        else:
            consulta = self.consulta.expand(x.shape[0], -1, -1)
            vector = self.decoder(consulta, h, None, mascara)[:, 0]
        return self.densa(vector)

    def forward(self, x, largos):
        return self.salida(self.representacion(x, largos))


def inicializar_pesos(m):
    if isinstance(m, (nn.Linear, nn.Embedding)):
        nn.init.xavier_uniform_(m.weight.data)


def construir_modelo(config, n_vocab, pad_idx, cls_idx):
    if config["familia"] == "recurrente":
        return ClasificadorRecurrente(config, n_vocab, pad_idx).to(device)
    modelo = ClasificadorTransformer(config, n_vocab, pad_idx, cls_idx)
    modelo.apply(inicializar_pesos)
    return modelo.to(device)

### 6.3 Configuraciones base

Los hiperparámetros compartidos se fijan iguales en ambas familias para que la comparación sea justa: `embed_dim = 128`, `maxlen = 320`, `min_freq = 5`, capa de representación de 128 y `clip = 1`.

**Red recurrente:**

| Hiperparámetro | Valor base | Justificación |
|---|---|---|
| `celda` | `RNN` | La más simple; la sección 9 la compara con LSTM y GRU |
| `units` | 128 | Estado oculto del mismo tamaño que el *embedding* |
| `capas` | 1 | Se prueba apilar en la sección 11 |
| `bidireccional` | `False` | Se prueba en la sección 11 |
| `dropout` | 0,25 | Las reseñas se memorizan pronto; regularización moderada, igual que en la CNN |
| `learning_rate` | 0,001 | Valor por defecto de Adam (Kingma y Ba, 2015) |
| `batch_size` | 64 | 543 actualizaciones por época con 34.708 reseñas |

**Transformer:**

| Hiperparámetro | Valor base | Justificación |
|---|---|---|
| `agregacion` | `promedio` | La forma más simple de resumir; la sección 16 la compara con `[CLS]` y con el decoder |
| `hid_dim` | 128 | Igual al *embedding*: no necesita proyección |
| `capas` | 2 | Encoder pequeño para ≈ 35.000 reseñas; se prueba en la sección 18 |
| `cabezas` | 4 | Cabezas de 32 dimensiones; se prueba en la sección 18 |
| `pf_dim` | 256 | El doble de `hid_dim` (el diseño original usa 4 veces) |
| `dropout` | 0,1 | Valor del diseño original (Vaswani et al., 2017) |
| `learning_rate` | 0,0005 | Menor que en la red recurrente: la atención es más sensible a pasos grandes sin un calentamiento de la tasa |
| `batch_size` | 64 | Igual que la red recurrente |

In [ ]:
# ============================================================
# Configuraciones base
# ============================================================
COMUNES = dict(embed_dim=128, maxlen=MAXLEN, min_freq=MIN_FREQ, clip=1.0)
CONFIG_BASE_RNN = dict(familia="recurrente", celda="RNN", units=128, capas=1, bidireccional=False, dropout=0.25,
                       learning_rate=0.001, batch_size=64, **COMUNES)
CONFIG_BASE_TRF = dict(familia="transformer", agregacion="promedio", hid_dim=128, capas=2, cabezas=4, pf_dim=256,
                       dropout=0.1, learning_rate=0.0005, batch_size=64, **COMUNES)
mostrar_config(CONFIG_BASE_RNN)

In [ ]:
# ============================================================
# Configuración base del Transformer
# ============================================================
mostrar_config(CONFIG_BASE_TRF)

# Parte A — Redes recurrentes

Las semillas adicionales del rango de error se definen aquí y se usan en ambas familias y en las comparaciones con 5 semillas.

In [ ]:
# ============================================================
# Semillas del rango de error (ambas familias)
# ============================================================
SEMILLAS_ADICIONALES = [1, 2, 3, 4]
print("Semillas:", [SEED] + SEMILLAS_ADICIONALES)

## 7. Entrenamiento del modelo base (recurrente)

Se entrena la configuración base con la semilla 42, mostrando el avance por época. Esta corrida es la referencia de los experimentos de la familia y la primera de las 5 semillas de su rango de error.

In [ ]:
# ============================================================
# Entrenamiento del modelo base (recurrente)
# ============================================================
fila_base_rnn, historia_base_rnn, modelo_base_rnn = entrenar_y_registrar(CONFIG_BASE_RNN, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Estructura, parámetros y métricas del modelo base (recurrente)
# ============================================================
print(modelo_base_rnn)
print(f"Parámetros entrenables: {contar_parametros(modelo_base_rnn):,}")
display(pd.DataFrame([fila_base_rnn]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo base (recurrente)
# ============================================================
graficar_curvas(historia_base_rnn, "Modelo base (recurrente)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro y exactitud de validación frente a las líneas base (sección 3.8), en particular frente a TF-IDF + regresión logística.
- Mejor época, brecha entrenamiento–validación y época en que la pérdida de validación empieza a subir.
- Segundos por época: costo de cada corrida de esta familia.

## 8. Rango de error entre reentrenamientos (recurrente)

Entrenar dos veces la misma configuración no entrega el mismo resultado: cambian los pesos iniciales, el orden de los lotes y el Dropout. Se reentrena la configuración base con otras cuatro semillas (5 en total) y se calcula el **rango** (máximo − mínimo) del F1 macro de validación. Ese valor queda en `MARGEN_ERROR_RNN` y se usa en todos los experimentos de esta familia: dos variantes cuya diferencia sea menor o igual se consideran en **empate técnico**.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración base (recurrente)
# ============================================================
filas_variabilidad_rnn = [{"semilla": SEED, **fila_base_rnn}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE_RNN, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad_rnn.append({"semilla": semilla, **fila})
tabla_variabilidad_rnn = pd.DataFrame(filas_variabilidad_rnn)

In [ ]:
# ============================================================
# Resultados por semilla (recurrente)
# ============================================================
display(tabla_variabilidad_rnn.round(4))

In [ ]:
# ============================================================
# Estadísticos de variación y rango de error (recurrente)
# ============================================================
columnas_variacion = ["f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca"]
estadisticos_variabilidad_rnn = tabla_variabilidad_rnn[columnas_variacion].agg(["mean", "std", "min", "max"]).T
estadisticos_variabilidad_rnn["rango"] = estadisticos_variabilidad_rnn["max"] - estadisticos_variabilidad_rnn["min"]
MARGEN_ERROR_RNN = estadisticos_variabilidad_rnn.loc["f1_macro_val", "rango"]
CADENA_RNN = nueva_cadena("recurrente", CONFIG_BASE_RNN, MARGEN_ERROR_RNN)
print(f"Rango de error (recurrente): {MARGEN_ERROR_RNN:.4f} ({MARGEN_ERROR_RNN * 100:.2f} pp)")
display(estadisticos_variabilidad_rnn.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Media, desviación estándar y rango del F1 macro con 5 semillas: el rango de error de la familia en puntos porcentuales.
- Variación de la mejor época entre semillas.

## 9. Experimento clave: tipo de celda (RNN simple, LSTM y GRU)

Es la decisión central de la familia recurrente. Con reseñas de cientos de tokens, se espera que la RNN simple pierda la señal del inicio por el desvanecimiento del gradiente y que las celdas con compuertas (LSTM y GRU) la conserven. Las tres variantes usan la configuración vigente; se entrenan siempre, para graficar sus curvas y analizar su desempeño según el largo de la reseña.

### 9.1 Comparación de celdas

Variantes: RNN simple, LSTM y GRU, con el mismo número de unidades.

In [ ]:
# ============================================================
# Experimento 9.1: comparación de celdas
# ============================================================
variantes_celda = {"RNN": dict(celda="RNN"), "LSTM": dict(celda="LSTM"), "GRU": dict(celda="GRU")}
tabla_celda = ejecutar_experimento(CADENA_RNN, "celda", variantes_celda, forzar=True)
historias_celda = dict(historias_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 9.1
# ============================================================
display(tabla_celda[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Curvas de validación del experimento 9.1
# ============================================================
graficar_comparacion(historias_celda, "f1_val", "F1 macro de validación por época según la celda", "F1 macro de validación")

In [ ]:
# ============================================================
# Selección de la variante del experimento 9.1
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_celda)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "celda", variantes_celda, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencia de F1 macro entre las celdas con compuertas y la RNN simple, comparada con el rango de error.
- Parámetros y segundos por época de cada celda: costo de las compuertas.
- Si LSTM y GRU empatan, opciones y criterio (se decide con el usuario).

### 9.2 Desempeño según el largo de la reseña

Si la RNN simple pierde la señal en secuencias largas, su desventaja debería crecer con el largo. Se calcula el F1 macro de validación de cada celda en los tres tramos de largo de la sección 5.4, con los modelos recién entrenados.

In [ ]:
# ============================================================
# F1 de validación por tramo de largo para cada celda
# ============================================================
tabla_tramos_celda = pd.concat([
    f1_por_largo(pred["y_true"], pred["y_pred"], DATOS_VAL["largo_original"]).assign(celda=celda)
    for celda, pred in ((c, predecir(m, DATOS_VAL)) for c, m in modelos_ultimo_experimento.items())
], ignore_index=True)
display(tabla_tramos_celda.pivot(index="tramo", columns="celda", values="f1_macro").loc[[t for _, _, t in TRAMOS]].round(4))

In [ ]:
# ============================================================
# Gráfico del F1 por tramo de largo
# ============================================================
fig = px.bar(tabla_tramos_celda, x="tramo", y="f1_macro", color="celda", barmode="group",
             color_discrete_sequence=COLORES, category_orders={"tramo": [t for _, _, t in TRAMOS]},
             labels={"f1_macro": "F1 macro (validación)", "tramo": "largo original de la reseña"}, height=420)
fig.update_layout(title="F1 macro de validación por largo de la reseña y tipo de celda")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿La brecha entre la RNN simple y las celdas con compuertas crece con el largo?
- Tramo en que cada celda rinde peor y relación con el truncado a 320 tokens.

## 10. Experimentos de entrenamiento (recurrente): tasa de aprendizaje, tamaño del lote y épocas

Son los tres hiperparámetros de entrenamiento que pide optimizar el problema.

### 10.1 Tasa de aprendizaje

La tasa define el tamaño de cada paso de actualización: muy baja, el modelo no converge dentro del tope; muy alta, la pérdida oscila o diverge. Se prueban 0,0003, 0,001 y 0,003 (pasos de ≈ 3 veces); si la mejor queda en un borde, se amplía hasta 0,0001 o 0,01.

In [ ]:
# ============================================================
# Experimento 10.1: tasa de aprendizaje
# ============================================================
variantes_rnn_tasa = {"0.0003": dict(learning_rate=0.0003), "0.001": dict(learning_rate=0.001), "0.003": dict(learning_rate=0.003)}
tabla_rnn_tasa = ejecutar_experimento(CADENA_RNN, "tasa de aprendizaje", variantes_rnn_tasa)
tabla_rnn_tasa = ampliar_rango(CADENA_RNN, "tasa de aprendizaje", variantes_rnn_tasa, "learning_rate", lambda v: round(v * 3, 6), lambda v: round(v / 3, 6), 0.0001, 0.01)

In [ ]:
# ============================================================
# Resultados del experimento 10.1
# ============================================================
display(tabla_rnn_tasa[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 10.1
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_rnn_tasa)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "tasa de aprendizaje", variantes_rnn_tasa, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 10.2 Tamaño del lote

Con 34.708 reseñas, un lote de 32 da 1.085 actualizaciones por época y uno de 128, 272. Los lotes pequeños actualizan más veces y con más ruido; los grandes aprovechan mejor la GPU pero pueden necesitar una tasa mayor. Se prueban 32, 64 y 128 con la tasa vigente; si la mejor queda en un borde, se amplía entre 16 y 512. La interacción entre lote y tasa se verifica en la sección 13.3.

In [ ]:
# ============================================================
# Experimento 10.2: tamaño del lote
# ============================================================
variantes_rnn_lote = {"32": dict(batch_size=32), "64": dict(batch_size=64), "128": dict(batch_size=128)}
tabla_rnn_lote = ejecutar_experimento(CADENA_RNN, "tamaño del lote", variantes_rnn_lote)
tabla_rnn_lote = ampliar_rango(CADENA_RNN, "tamaño del lote", variantes_rnn_lote, "batch_size", lambda v: v * 2, lambda v: v // 2, 16, 512)

In [ ]:
# ============================================================
# Resultados del experimento 10.2
# ============================================================
display(tabla_rnn_lote[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 10.2
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_rnn_lote)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "tamaño del lote", variantes_rnn_lote, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 10.3 Épocas sin Early Stopping

Para justificar el tope de 15 épocas y la paciencia de 3, se entrena la configuración vigente de la familia durante 20 épocas **sin Early Stopping** y se observa en qué época la pérdida de validación deja de bajar y empieza a subir (inicio del sobreajuste). Esta corrida no cambia la configuración.

In [ ]:
# ============================================================
# Experimento 10.3: 20 épocas sin Early Stopping (recurrente)
# ============================================================
EPOCAS_SIN_ES = 20
fila_epocas_rnn, historia_epocas_rnn, _ = entrenar_y_registrar(CADENA_RNN["config"], epocas=EPOCAS_SIN_ES, paciencia=0, forzar=True)

In [ ]:
# ============================================================
# Curvas sin Early Stopping (recurrente)
# ============================================================
graficar_curvas(historia_epocas_rnn, "recurrente: 20 épocas sin Early Stopping")

In [ ]:
# ============================================================
# Métricas de validación en épocas seleccionadas (recurrente)
# ============================================================
epocas_revisadas = [1, 2, 3, 5, 10, 15, 20]
tabla_epocas_rnn = pd.DataFrame({k: [historia_epocas_rnn[k][e - 1] for e in epocas_revisadas]
                                     for k in ["loss_train", "loss_val", "exactitud_val", "f1_val"]}, index=epocas_revisadas)
tabla_epocas_rnn.index.name = "época"
print(f"Mejor época por pérdida de validación: {int(np.argmin(historia_epocas_rnn['loss_val'])) + 1}")
display(tabla_epocas_rnn.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Época de menor pérdida de validación y desde cuándo sube (sobreajuste).
- ¿El F1 de validación cae junto con la subida de la pérdida o se mantiene?
- ¿El tope de 15 épocas con paciencia 3 es suficiente para esta familia?

## 11. Experimentos de arquitectura (recurrente)

Se ajusta la capacidad de la red recurrente: tamaño del estado oculto, profundidad y sentido de lectura.

### 11.1 Unidades del estado oculto (`units`)

Más unidades permiten resumir más información de la reseña en el estado oculto, a cambio de más parámetros (en una LSTM, 4 × (entrada + unidades) × unidades por capa). Se prueban 64, 128 y 256; si la mejor queda en un borde, se amplía entre 32 y 512.

In [ ]:
# ============================================================
# Experimento 11.1: unidades del estado oculto (`units`)
# ============================================================
variantes_units = {"64": dict(units=64), "128": dict(units=128), "256": dict(units=256)}
tabla_units = ejecutar_experimento(CADENA_RNN, "units", variantes_units)
tabla_units = ampliar_rango(CADENA_RNN, "units", variantes_units, "units", lambda v: v * 2, lambda v: v // 2, 32, 512)

In [ ]:
# ============================================================
# Resultados del experimento 11.1
# ============================================================
display(tabla_units[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.1
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_units)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "units", variantes_units, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 11.2 Número de capas recurrentes (`capas`)

Apilar capas permite que la segunda capa lea la secuencia de estados de la primera y capture patrones de más alto nivel. Con más de una capa, el Dropout se aplica también entre capas. Se prueban 1, 2 y 3; si gana 3, se amplía a 4.

In [ ]:
# ============================================================
# Experimento 11.2: número de capas recurrentes (`capas`)
# ============================================================
variantes_rnn_capas = {"1": dict(capas=1), "2": dict(capas=2), "3": dict(capas=3)}
tabla_rnn_capas = ejecutar_experimento(CADENA_RNN, "capas", variantes_rnn_capas)
tabla_rnn_capas = ampliar_rango(CADENA_RNN, "capas", variantes_rnn_capas, "capas", lambda v: v + 1, lambda v: v - 1, 1, 4)

In [ ]:
# ============================================================
# Resultados del experimento 11.2
# ============================================================
display(tabla_rnn_capas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.2
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_rnn_capas)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "capas", variantes_rnn_capas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 11.3 Lectura bidireccional (`bidireccional`)

Una red unidireccional llega al final de la reseña con un estado que privilegia lo último que leyó; leyendo también de atrás hacia adelante, el estado final del sentido inverso resume el inicio. Duplica el costo por época.

In [ ]:
# ============================================================
# Experimento 11.3: lectura bidireccional (`bidireccional`)
# ============================================================
variantes_bidireccional = {"unidireccional": dict(bidireccional=False), "bidireccional": dict(bidireccional=True)}
tabla_bidireccional = ejecutar_experimento(CADENA_RNN, "bidireccional", variantes_bidireccional)

In [ ]:
# ============================================================
# Resultados del experimento 11.3
# ============================================================
display(tabla_bidireccional[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.3
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_bidireccional)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "bidireccional", variantes_bidireccional, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 12. Experimento de regularización (recurrente)

El desbalance no requiere corrección (clases balanceadas, sección 3.3), así que la regularización se concentra en el Dropout.

### 12.1 Dropout

El Dropout apaga al azar una fracción de las neuronas en cada paso de entrenamiento (Srivastava et al., 2014); aquí se aplica antes de la capa de representación, antes de la salida y, con más de una capa, entre capas recurrentes. Se prueban 0; 0,25 y 0,5; si gana 0,5, se amplía a 0,75.

In [ ]:
# ============================================================
# Experimento 12.1: dropout
# ============================================================
variantes_rnn_dropout = {"0.0": dict(dropout=0.0), "0.25": dict(dropout=0.25), "0.5": dict(dropout=0.5)}
tabla_rnn_dropout = ejecutar_experimento(CADENA_RNN, "dropout", variantes_rnn_dropout)
tabla_rnn_dropout = ampliar_rango(CADENA_RNN, "dropout", variantes_rnn_dropout, "dropout", lambda v: v + 0.25, lambda v: v - 0.25, 0.0, 0.75)

In [ ]:
# ============================================================
# Resultados del experimento 12.1
# ============================================================
display(tabla_rnn_dropout[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 12.1
# ============================================================
elegida = seleccionar_variante(CADENA_RNN, tabla_rnn_dropout)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_RNN, "dropout", variantes_rnn_dropout, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 13. Modelo final (recurrente)

### 13.1 Configuración final y tabla de decisiones

La configuración final es la que quedó en la cadena de la familia después de todos sus experimentos. La tabla de decisiones muestra, para cada experimento, la variante elegida, su F1 macro de validación y el de la variante que coincidía con la configuración previa, para ver cuánto aportó cada decisión y si supera el rango de error.

In [ ]:
# ============================================================
# Configuración final (recurrente)
# ============================================================
CONFIG_FINAL_RNN = dict(CADENA_RNN["config"])
mostrar_config(CONFIG_FINAL_RNN)

In [ ]:
# ============================================================
# Tabla de decisiones (recurrente)
# ============================================================
tabla_decisiones_rnn = pd.DataFrame.from_dict(CADENA_RNN["decisiones"], orient="index")
tabla_decisiones_rnn.index.name = "experimento"
tabla_decisiones_rnn["aporte"] = tabla_decisiones_rnn["f1_elegida"] - tabla_decisiones_rnn["f1_previa"]
tabla_decisiones_rnn["supera_margen"] = tabla_decisiones_rnn["aporte"].abs() > CADENA_RNN["margen"]
tabla_decisiones_rnn = tabla_decisiones_rnn.reset_index()
display(tabla_decisiones_rnn.round(4))

### 13.2 Entrenamiento del modelo final

Se entrena la configuración final con la semilla 42 y el mismo régimen de épocas que los experimentos. Esta corrida se entrena siempre, porque su modelo se evalúa en test y puede pasar a la fusión.

In [ ]:
# ============================================================
# Entrenamiento del modelo final (recurrente)
# ============================================================
fila_final_rnn, historia_final_rnn, modelo_final_rnn = entrenar_y_registrar(CONFIG_FINAL_RNN, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Métricas del modelo final en su mejor época (recurrente)
# ============================================================
print(f"Parámetros entrenables: {contar_parametros(modelo_final_rnn):,}")
display(pd.DataFrame([fila_final_rnn]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo final (recurrente)
# ============================================================
graficar_curvas(historia_final_rnn, "Modelo final (recurrente)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de validación del modelo final frente al base y frente a la media de las 5 semillas del base.
- Mejor época y brecha entrenamiento–validación frente al base.

### 13.3 Verificaciones de la dependencia del orden

Elegir un hiperparámetro a la vez hace que cada decisión dependa de las anteriores. Sin usar test, se prueban sobre la configuración final:

- **Tasa × lote:** al duplicar el lote, la tasa óptima suele subir, porque el gradiente promedio es menos ruidoso. Se prueban ambos duplicados a la vez.
- **Capacidad × regularización:** una red más grande puede necesitar más regularización. Se prueban el doble de unidades con 0,25 más de Dropout.
- **Celda:** la celda se eligió al principio, con la configuración base; se prueba la segunda mejor del experimento 9.1 con la configuración final.

Una diferencia mayor que el rango de error indica que el orden de las decisiones sí condicionó el resultado. Los experimentos ya hechos no se rehacen: se reinterpretan con estos resultados.

In [ ]:
# ============================================================
# Verificaciones del orden (recurrente)
# ============================================================
VERIFICACIONES_RNN = {
    "13.3a Tasa ×2 con lote ×2": dict(learning_rate=CONFIG_FINAL_RNN["learning_rate"] * 2, batch_size=CONFIG_FINAL_RNN["batch_size"] * 2),
    "13.3b Units ×2 con Dropout +0,25": dict(units=CONFIG_FINAL_RNN["units"] * 2, dropout=min(0.75, CONFIG_FINAL_RNN["dropout"] + 0.25)),
    "13.3c Segunda mejor celda con la configuración final": dict(celda=next(v for v in tabla_celda["variante"] if v != CONFIG_FINAL_RNN["celda"])),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final_rnn}]
for nombre, cambios in VERIFICACIONES_RNN.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL_RNN, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion_rnn = pd.DataFrame(filas_verificacion)
tabla_verificacion_rnn["diferencia_vs_final"] = tabla_verificacion_rnn["f1_macro_val"] - fila_final_rnn["f1_macro_val"]
tabla_verificacion_rnn["supera_margen"] = tabla_verificacion_rnn["diferencia_vs_final"].abs() > CADENA_RNN["margen"]

In [ ]:
# ============================================================
# Resultados de las verificaciones (recurrente)
# ============================================================
display(tabla_verificacion_rnn[["verificacion", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val",
                                    "mejor_epoca", "parametros", "segundos_por_epoca", "diferencia_vs_final",
                                    "supera_margen", "origen"]].round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Alguna combinación supera a la configuración final por más que el rango de error?
- Si es así, qué decisión quedó condicionada por el orden (se decide con el usuario si se adopta).

### 13.4 Configuración final con 5 semillas

Se reentrena la configuración final con las mismas semillas adicionales del rango de error y se compara su distribución de F1 macro con la del modelo base de la familia. Esta media es además el criterio para elegir el modelo de texto que pasa a la fusión (sección 25).

In [ ]:
# ============================================================
# Reentrenamientos de la configuración final (recurrente)
# ============================================================
filas_final_semillas_rnn = [{"semilla": SEED, **fila_final_rnn}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL_RNN, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas_rnn.append({"semilla": semilla, **fila})
tabla_final_semillas_rnn = pd.DataFrame(filas_final_semillas_rnn)

In [ ]:
# ============================================================
# Comparación justa: base frente a final con 5 semillas (recurrente)
# ============================================================
comparacion_justa_rnn = pd.DataFrame({
    "base": tabla_variabilidad_rnn["f1_macro_val"].agg(["mean", "std", "min", "max"]),
    "final": tabla_final_semillas_rnn["f1_macro_val"].agg(["mean", "std", "min", "max"]),
})
comparacion_justa_rnn["diferencia"] = comparacion_justa_rnn["final"] - comparacion_justa_rnn["base"]
print(f"Mejora media: {comparacion_justa_rnn.loc['mean', 'diferencia'] * 100:.2f} pp | rango de error: {CADENA_RNN['margen'] * 100:.2f} pp")
display(comparacion_justa_rnn.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mejora media de F1 macro con 5 semillas frente a la estimación de una sola semilla (cota optimista).
- ¿Los rangos de base y final se superponen?
- Advertencia: muchas variantes se evaluaron sobre la misma validación, así que su F1 final es optimista; el test es la estimación independiente.

# Parte B — Transformer

## 14. Entrenamiento del modelo base (Transformer)

Se entrena la configuración base con la semilla 42, mostrando el avance por época. Esta corrida es la referencia de los experimentos de la familia y la primera de las 5 semillas de su rango de error.

In [ ]:
# ============================================================
# Entrenamiento del modelo base (Transformer)
# ============================================================
fila_base_trf, historia_base_trf, modelo_base_trf = entrenar_y_registrar(CONFIG_BASE_TRF, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Estructura, parámetros y métricas del modelo base (Transformer)
# ============================================================
print(modelo_base_trf)
print(f"Parámetros entrenables: {contar_parametros(modelo_base_trf):,}")
display(pd.DataFrame([fila_base_trf]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo base (Transformer)
# ============================================================
graficar_curvas(historia_base_trf, "Modelo base (Transformer)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro y exactitud de validación frente a las líneas base (sección 3.8), en particular frente a TF-IDF + regresión logística.
- Mejor época, brecha entrenamiento–validación y época en que la pérdida de validación empieza a subir.
- Segundos por época: costo de cada corrida de esta familia.

## 15. Rango de error entre reentrenamientos (Transformer)

Entrenar dos veces la misma configuración no entrega el mismo resultado: cambian los pesos iniciales, el orden de los lotes y el Dropout. Se reentrena la configuración base con otras cuatro semillas (5 en total) y se calcula el **rango** (máximo − mínimo) del F1 macro de validación. Ese valor queda en `MARGEN_ERROR_TRF` y se usa en todos los experimentos de esta familia: dos variantes cuya diferencia sea menor o igual se consideran en **empate técnico**.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración base (Transformer)
# ============================================================
filas_variabilidad_trf = [{"semilla": SEED, **fila_base_trf}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE_TRF, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad_trf.append({"semilla": semilla, **fila})
tabla_variabilidad_trf = pd.DataFrame(filas_variabilidad_trf)

In [ ]:
# ============================================================
# Resultados por semilla (Transformer)
# ============================================================
display(tabla_variabilidad_trf.round(4))

In [ ]:
# ============================================================
# Estadísticos de variación y rango de error (Transformer)
# ============================================================
columnas_variacion = ["f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca"]
estadisticos_variabilidad_trf = tabla_variabilidad_trf[columnas_variacion].agg(["mean", "std", "min", "max"]).T
estadisticos_variabilidad_trf["rango"] = estadisticos_variabilidad_trf["max"] - estadisticos_variabilidad_trf["min"]
MARGEN_ERROR_TRF = estadisticos_variabilidad_trf.loc["f1_macro_val", "rango"]
CADENA_TRF = nueva_cadena("Transformer", CONFIG_BASE_TRF, MARGEN_ERROR_TRF)
print(f"Rango de error (Transformer): {MARGEN_ERROR_TRF:.4f} ({MARGEN_ERROR_TRF * 100:.2f} pp)")
display(estadisticos_variabilidad_trf.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Media, desviación estándar y rango del F1 macro con 5 semillas: el rango de error de la familia en puntos porcentuales.
- Variación de la mejor época entre semillas.

## 16. Experimento clave: cómo se resume la reseña (encoder, `[CLS]` o decoder)

El encoder entrega un vector por token; para clasificar hay que resumir la reseña en un solo vector. Se comparan las tres formas de la sección 6.2: el **promedio** de las salidas del encoder, el token **`[CLS]`** y un **decoder de un paso** con atención cruzada. El promedio trata a todos los tokens por igual; `[CLS]` y el decoder aprenden a qué tokens dar más peso, y el decoder además deja visibles esos pesos (sección 22.2).

La variante ganadora queda en la configuración y se usa en todos los experimentos siguientes y en el modelo final; si hay empate técnico, se decide con el usuario. Las tres variantes se entrenan siempre, para graficar sus curvas. La elección se verifica de nuevo con la configuración final en la sección 20.3.

### 16.1 Forma de resumir la reseña

Variantes: promedio, `[CLS]` y decoder de un paso, con la configuración vigente.

In [ ]:
# ============================================================
# Experimento 16.1: forma de resumir la reseña
# ============================================================
variantes_agregacion = {"promedio": dict(agregacion="promedio"), "[CLS]": dict(agregacion="cls"), "decoder": dict(agregacion="decoder")}
tabla_agregacion = ejecutar_experimento(CADENA_TRF, "agregación", variantes_agregacion, forzar=True)
historias_agregacion = dict(historias_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 16.1
# ============================================================
display(tabla_agregacion[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Curvas de validación del experimento 16.1
# ============================================================
graficar_comparacion(historias_agregacion, "f1_val", "F1 macro de validación por época según la forma de resumir", "F1 macro de validación")

In [ ]:
# ============================================================
# Selección de la variante del experimento 16.1
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_agregacion)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "agregación", variantes_agregacion, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencia de F1 macro entre las tres formas, comparada con el rango de error del Transformer.
- Parámetros y segundos por época que agrega el decoder.
- Si hay empate técnico, opciones y criterio (se decide con el usuario).

## 17. Experimentos de entrenamiento (Transformer): tasa de aprendizaje, tamaño del lote y épocas

Los mismos tres hiperparámetros de entrenamiento de la red recurrente, con su propia cadena de decisiones.

### 17.1 Tasa de aprendizaje

Se prueban 0,00025, 0,0005 y 0,001 (pasos de 2 veces, más finos que en la red recurrente porque la atención sin calentamiento de la tasa es sensible a pasos grandes); si la mejor queda en un borde, se amplía hasta 0,0000625 o 0,004.

In [ ]:
# ============================================================
# Experimento 17.1: tasa de aprendizaje
# ============================================================
variantes_trf_tasa = {"0.00025": dict(learning_rate=0.00025), "0.0005": dict(learning_rate=0.0005), "0.001": dict(learning_rate=0.001)}
tabla_trf_tasa = ejecutar_experimento(CADENA_TRF, "tasa de aprendizaje", variantes_trf_tasa)
tabla_trf_tasa = ampliar_rango(CADENA_TRF, "tasa de aprendizaje", variantes_trf_tasa, "learning_rate", lambda v: round(v * 2, 7), lambda v: round(v / 2, 7), 0.0000625, 0.004)

In [ ]:
# ============================================================
# Resultados del experimento 17.1
# ============================================================
display(tabla_trf_tasa[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 17.1
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_trf_tasa)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "tasa de aprendizaje", variantes_trf_tasa, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 17.2 Tamaño del lote

Se prueban 32, 64 y 128 con la tasa vigente; si la mejor queda en un borde, se amplía entre 16 y 512. En el Transformer, la memoria de la atención crece con lote × cabezas × largo², así que los lotes grandes con reseñas de 320 tokens son los más exigentes en memoria.

In [ ]:
# ============================================================
# Experimento 17.2: tamaño del lote
# ============================================================
variantes_trf_lote = {"32": dict(batch_size=32), "64": dict(batch_size=64), "128": dict(batch_size=128)}
tabla_trf_lote = ejecutar_experimento(CADENA_TRF, "tamaño del lote", variantes_trf_lote)
tabla_trf_lote = ampliar_rango(CADENA_TRF, "tamaño del lote", variantes_trf_lote, "batch_size", lambda v: v * 2, lambda v: v // 2, 16, 512)

In [ ]:
# ============================================================
# Resultados del experimento 17.2
# ============================================================
display(tabla_trf_lote[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 17.2
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_trf_lote)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "tamaño del lote", variantes_trf_lote, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 17.3 Épocas sin Early Stopping

Para justificar el tope de 15 épocas y la paciencia de 3, se entrena la configuración vigente de la familia durante 20 épocas **sin Early Stopping** y se observa en qué época la pérdida de validación deja de bajar y empieza a subir (inicio del sobreajuste). Esta corrida no cambia la configuración.

In [ ]:
# ============================================================
# Experimento 17.3: 20 épocas sin Early Stopping (Transformer)
# ============================================================
EPOCAS_SIN_ES = 20
fila_epocas_trf, historia_epocas_trf, _ = entrenar_y_registrar(CADENA_TRF["config"], epocas=EPOCAS_SIN_ES, paciencia=0, forzar=True)

In [ ]:
# ============================================================
# Curvas sin Early Stopping (Transformer)
# ============================================================
graficar_curvas(historia_epocas_trf, "Transformer: 20 épocas sin Early Stopping")

In [ ]:
# ============================================================
# Métricas de validación en épocas seleccionadas (Transformer)
# ============================================================
epocas_revisadas = [1, 2, 3, 5, 10, 15, 20]
tabla_epocas_trf = pd.DataFrame({k: [historia_epocas_trf[k][e - 1] for e in epocas_revisadas]
                                     for k in ["loss_train", "loss_val", "exactitud_val", "f1_val"]}, index=epocas_revisadas)
tabla_epocas_trf.index.name = "época"
print(f"Mejor época por pérdida de validación: {int(np.argmin(historia_epocas_trf['loss_val'])) + 1}")
display(tabla_epocas_trf.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Época de menor pérdida de validación y desde cuándo sube (sobreajuste).
- ¿El F1 de validación cae junto con la subida de la pérdida o se mantiene?
- ¿El tope de 15 épocas con paciencia 3 es suficiente para esta familia?

## 18. Experimentos de arquitectura (Transformer)

Se ajusta la capacidad del Transformer: profundidad del encoder, número de cabezas, dimensión de los vectores y tamaño de la red *feed-forward*.

### 18.1 Número de bloques encoder (`capas`)

Cada bloque permite combinar la información que ya reunió el bloque anterior. Se prueban 1, 2 y 3; si gana 3, se amplía a 4.

In [ ]:
# ============================================================
# Experimento 18.1: número de bloques encoder (`capas`)
# ============================================================
variantes_trf_capas = {"1": dict(capas=1), "2": dict(capas=2), "3": dict(capas=3)}
tabla_trf_capas = ejecutar_experimento(CADENA_TRF, "capas", variantes_trf_capas)
tabla_trf_capas = ampliar_rango(CADENA_TRF, "capas", variantes_trf_capas, "capas", lambda v: v + 1, lambda v: v - 1, 1, 4)

In [ ]:
# ============================================================
# Resultados del experimento 18.1
# ============================================================
display(tabla_trf_capas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 18.1
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_trf_capas)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "capas", variantes_trf_capas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 18.2 Número de cabezas de atención (`cabezas`)

Con `hid_dim` fijo, más cabezas significan más tipos de relación entre tokens, pero cada una con menos dimensiones (`hid_dim / cabezas`). Se prueban 2, 4 y 8.

In [ ]:
# ============================================================
# Experimento 18.2: número de cabezas de atención (`cabezas`)
# ============================================================
variantes_cabezas = {"2": dict(cabezas=2), "4": dict(cabezas=4), "8": dict(cabezas=8)}
tabla_cabezas = ejecutar_experimento(CADENA_TRF, "cabezas", variantes_cabezas)

In [ ]:
# ============================================================
# Resultados del experimento 18.2
# ============================================================
display(tabla_cabezas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 18.2
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_cabezas)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "cabezas", variantes_cabezas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 18.3 Dimensión de los vectores (`hid_dim`)

`hid_dim` es el tamaño del vector de cada token dentro del encoder. Si difiere de `embed_dim` (128), una capa lineal proyecta el *embedding*. Se prueban 64, 128 y 256; si la mejor queda en un borde, se amplía entre 32 y 512. Todos los valores son divisibles por las cabezas probadas.

In [ ]:
# ============================================================
# Experimento 18.3: dimensión de los vectores (`hid_dim`)
# ============================================================
variantes_hid_dim = {"64": dict(hid_dim=64), "128": dict(hid_dim=128), "256": dict(hid_dim=256)}
tabla_hid_dim = ejecutar_experimento(CADENA_TRF, "hid_dim", variantes_hid_dim)
tabla_hid_dim = ampliar_rango(CADENA_TRF, "hid_dim", variantes_hid_dim, "hid_dim", lambda v: v * 2, lambda v: v // 2, 32, 512)

In [ ]:
# ============================================================
# Resultados del experimento 18.3
# ============================================================
display(tabla_hid_dim[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 18.3
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_hid_dim)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "hid_dim", variantes_hid_dim, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 18.4 Tamaño de la red feed-forward (`pf_dim`)

La red *feed-forward* de cada bloque procesa cada token por separado después de la atención. Se prueban 128, 256 y 512; si la mejor queda en un borde, se amplía entre 64 y 1.024.

In [ ]:
# ============================================================
# Experimento 18.4: tamaño de la red feed-forward (`pf_dim`)
# ============================================================
variantes_pf_dim = {"128": dict(pf_dim=128), "256": dict(pf_dim=256), "512": dict(pf_dim=512)}
tabla_pf_dim = ejecutar_experimento(CADENA_TRF, "pf_dim", variantes_pf_dim)
tabla_pf_dim = ampliar_rango(CADENA_TRF, "pf_dim", variantes_pf_dim, "pf_dim", lambda v: v * 2, lambda v: v // 2, 64, 1024)

In [ ]:
# ============================================================
# Resultados del experimento 18.4
# ============================================================
display(tabla_pf_dim[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 18.4
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_pf_dim)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "pf_dim", variantes_pf_dim, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 19. Experimento de regularización (Transformer)

Como en la red recurrente, la regularización se concentra en el Dropout, que en el Transformer se aplica a los *embeddings*, a los pesos de atención, a la red *feed-forward* y antes de la representación y la salida.

### 19.1 Dropout

Se prueban 0; 0,1 y 0,2; si gana 0,2, se amplía en pasos de 0,1 hasta 0,5.

In [ ]:
# ============================================================
# Experimento 19.1: dropout
# ============================================================
variantes_trf_dropout = {"0.0": dict(dropout=0.0), "0.1": dict(dropout=0.1), "0.2": dict(dropout=0.2)}
tabla_trf_dropout = ejecutar_experimento(CADENA_TRF, "dropout", variantes_trf_dropout)
tabla_trf_dropout = ampliar_rango(CADENA_TRF, "dropout", variantes_trf_dropout, "dropout", lambda v: round(v + 0.1, 2), lambda v: round(v - 0.1, 2), 0.0, 0.5)

In [ ]:
# ============================================================
# Resultados del experimento 19.1
# ============================================================
display(tabla_trf_dropout[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 19.1
# ============================================================
elegida = seleccionar_variante(CADENA_TRF, tabla_trf_dropout)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante(CADENA_TRF, "dropout", variantes_trf_dropout, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera el rango de error de la familia?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 20. Modelo final (Transformer)

### 20.1 Configuración final y tabla de decisiones

La configuración final es la que quedó en la cadena de la familia después de todos sus experimentos. La tabla de decisiones muestra, para cada experimento, la variante elegida, su F1 macro de validación y el de la variante que coincidía con la configuración previa, para ver cuánto aportó cada decisión y si supera el rango de error.

In [ ]:
# ============================================================
# Configuración final (Transformer)
# ============================================================
CONFIG_FINAL_TRF = dict(CADENA_TRF["config"])
mostrar_config(CONFIG_FINAL_TRF)

In [ ]:
# ============================================================
# Tabla de decisiones (Transformer)
# ============================================================
tabla_decisiones_trf = pd.DataFrame.from_dict(CADENA_TRF["decisiones"], orient="index")
tabla_decisiones_trf.index.name = "experimento"
tabla_decisiones_trf["aporte"] = tabla_decisiones_trf["f1_elegida"] - tabla_decisiones_trf["f1_previa"]
tabla_decisiones_trf["supera_margen"] = tabla_decisiones_trf["aporte"].abs() > CADENA_TRF["margen"]
tabla_decisiones_trf = tabla_decisiones_trf.reset_index()
display(tabla_decisiones_trf.round(4))

### 20.2 Entrenamiento del modelo final

Se entrena la configuración final con la semilla 42 y el mismo régimen de épocas que los experimentos. Esta corrida se entrena siempre, porque su modelo se evalúa en test y puede pasar a la fusión.

In [ ]:
# ============================================================
# Entrenamiento del modelo final (Transformer)
# ============================================================
fila_final_trf, historia_final_trf, modelo_final_trf = entrenar_y_registrar(CONFIG_FINAL_TRF, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Métricas del modelo final en su mejor época (Transformer)
# ============================================================
print(f"Parámetros entrenables: {contar_parametros(modelo_final_trf):,}")
display(pd.DataFrame([fila_final_trf]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo final (Transformer)
# ============================================================
graficar_curvas(historia_final_trf, "Modelo final (Transformer)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de validación del modelo final frente al base y frente a la media de las 5 semillas del base.
- Mejor época y brecha entrenamiento–validación frente al base.

### 20.3 Verificaciones de la dependencia del orden

Sin usar test, se prueban sobre la configuración final:

- **Tasa × lote:** ambos duplicados a la vez.
- **Capacidad × regularización:** un bloque encoder más con 0,1 más de Dropout.
- **Forma de resumir:** se eligió al principio, con la configuración base; se prueba la segunda mejor del experimento 16.1 con la configuración final.

Una diferencia mayor que el rango de error indica que el orden de las decisiones sí condicionó el resultado. Los experimentos ya hechos no se rehacen: se reinterpretan con estos resultados.

In [ ]:
# ============================================================
# Verificaciones del orden (Transformer)
# ============================================================
VERIFICACIONES_TRF = {
    "20.3a Tasa ×2 con lote ×2": dict(learning_rate=CONFIG_FINAL_TRF["learning_rate"] * 2, batch_size=CONFIG_FINAL_TRF["batch_size"] * 2),
    "20.3b Un bloque más con Dropout +0,1": dict(capas=CONFIG_FINAL_TRF["capas"] + 1, dropout=min(0.5, round(CONFIG_FINAL_TRF["dropout"] + 0.1, 2))),
    "20.3c Segunda mejor forma de resumir con la configuración final": dict(agregacion=next(variantes_agregacion[v]["agregacion"] for v in tabla_agregacion["variante"] if variantes_agregacion[v]["agregacion"] != CONFIG_FINAL_TRF["agregacion"])),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final_trf}]
for nombre, cambios in VERIFICACIONES_TRF.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL_TRF, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion_trf = pd.DataFrame(filas_verificacion)
tabla_verificacion_trf["diferencia_vs_final"] = tabla_verificacion_trf["f1_macro_val"] - fila_final_trf["f1_macro_val"]
tabla_verificacion_trf["supera_margen"] = tabla_verificacion_trf["diferencia_vs_final"].abs() > CADENA_TRF["margen"]

In [ ]:
# ============================================================
# Resultados de las verificaciones (Transformer)
# ============================================================
display(tabla_verificacion_trf[["verificacion", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val",
                                    "mejor_epoca", "parametros", "segundos_por_epoca", "diferencia_vs_final",
                                    "supera_margen", "origen"]].round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Alguna combinación supera a la configuración final por más que el rango de error?
- Si es así, qué decisión quedó condicionada por el orden (se decide con el usuario si se adopta).

### 20.4 Configuración final con 5 semillas

Se reentrena la configuración final con las mismas semillas adicionales del rango de error y se compara su distribución de F1 macro con la del modelo base de la familia. Esta media es además el criterio para elegir el modelo de texto que pasa a la fusión (sección 25).

In [ ]:
# ============================================================
# Reentrenamientos de la configuración final (Transformer)
# ============================================================
filas_final_semillas_trf = [{"semilla": SEED, **fila_final_trf}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL_TRF, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas_trf.append({"semilla": semilla, **fila})
tabla_final_semillas_trf = pd.DataFrame(filas_final_semillas_trf)

In [ ]:
# ============================================================
# Comparación justa: base frente a final con 5 semillas (Transformer)
# ============================================================
comparacion_justa_trf = pd.DataFrame({
    "base": tabla_variabilidad_trf["f1_macro_val"].agg(["mean", "std", "min", "max"]),
    "final": tabla_final_semillas_trf["f1_macro_val"].agg(["mean", "std", "min", "max"]),
})
comparacion_justa_trf["diferencia"] = comparacion_justa_trf["final"] - comparacion_justa_trf["base"]
print(f"Mejora media: {comparacion_justa_trf.loc['mean', 'diferencia'] * 100:.2f} pp | rango de error: {CADENA_TRF['margen'] * 100:.2f} pp")
display(comparacion_justa_trf.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mejora media de F1 macro con 5 semillas frente a la estimación de una sola semilla (cota optimista).
- ¿Los rangos de base y final se superponen?
- Advertencia: muchas variantes se evaluaron sobre la misma validación, así que su F1 final es optimista; el test es la estimación independiente.

## 21. Evaluación de los modelos finales en el conjunto de test

El conjunto de test (7.437 reseñas) no se usó en ninguna decisión. Se evalúa una sola vez con el modelo final de cada familia. Qué indica cada métrica en este problema:

- **Exactitud:** proporción de reseñas bien clasificadas; con clases balanceadas es una medida directa del desempeño.
- **Precisión de la clase positiva:** de las reseñas que el modelo llama positivas, cuántas lo son. Una precisión baja indica que el modelo **sobrepredice** el sentimiento positivo.
- **Exhaustividad de la clase positiva:** de las reseñas positivas reales, cuántas detecta. Una exhaustividad baja indica que confunde reseñas positivas con negativas.
- **F1 macro:** promedio del F1 de ambas clases; baja si una de las dos se resuelve peor.

Una clase solo se califica como bien resuelta si su precisión y su exhaustividad son altas a la vez.

### 21.1 Métricas globales frente a las líneas base

Se comparan ambos modelos finales con las líneas base de la sección 3.8, calculadas también en test.

In [ ]:
# ============================================================
# Predicción en test y métricas globales
# ============================================================
tokens_test = imdb["tokens"].to_numpy()[idx_test]
pred_test = {"recurrente": predecir(modelo_final_rnn, DATOS_TEST, CRITERIO),
             "Transformer": predecir(modelo_final_trf, DATOS_TEST, CRITERIO)}
pred_val = {"recurrente": predecir(modelo_final_rnn, DATOS_VAL, CRITERIO),
            "Transformer": predecir(modelo_final_trf, DATOS_VAL, CRITERIO)}
tabla_test = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(y_test, np.full(len(y_test), CLASE_MAYORITARIA))},
    {"modelo": "Regla léxica", **metricas(y_test, predecir_regla(tokens_test))},
    {"modelo": "TF-IDF + regresión logística", **metricas(y_test, regresion.predict(tfidf.transform(tokens_test)))},
    *[{"modelo": f"{familia} final", **metricas(p["y_true"], p["y_pred"]), "loss": p["perdida"]}
      for familia, p in pred_test.items()],
])
display(tabla_test.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Exactitud y F1 macro de cada modelo final en test frente a las líneas base (diferencia en puntos porcentuales).
- Diferencia entre validación y test de cada familia: ¿el test confirma lo estimado en validación?

### 21.2 Métricas por clase

Se muestran la precisión, la exhaustividad y el F1 de cada clase en test y en validación, para no sacar conclusiones por clase de un solo conjunto.

`reporte_por_clase(y_true, y_pred)` retorna una tabla con precisión, exhaustividad, F1 y soporte por clase.

In [ ]:
# ============================================================
# Reporte por clase en test y validación
# ============================================================
def reporte_por_clase(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    return pd.DataFrame({"precision": p, "exhaustividad": r, "f1": f, "soporte": s}, index=SENTIMIENTOS)


reporte = pd.concat({(familia, conjunto): reporte_por_clase(p["y_true"], p["y_pred"])
                     for conjunto, preds in [("test", pred_test), ("validación", pred_val)]
                     for familia, p in preds.items()})
display(reporte.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clase con menor F1 en cada familia y si coincide en test y validación.
- ¿Algún modelo sobrepredice una clase (precisión baja con exhaustividad alta)?

### 21.3 Matrices de confusión

Cada fila es la clase real y cada columna la predicha. Los valores fuera de la diagonal muestran en qué dirección se equivoca cada modelo.

In [ ]:
# ============================================================
# Matrices de confusión en test
# ============================================================
matrices = {familia: confusion_matrix(p["y_true"], p["y_pred"], labels=[0, 1]) for familia, p in pred_test.items()}
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{f} final" for f in matrices])
for col, matriz in enumerate(matrices.values(), start=1):
    fig.add_trace(go.Heatmap(z=matriz, x=SENTIMIENTOS, y=SENTIMIENTOS, text=matriz, texttemplate="%{text}",
                             colorscale=[[0, "#FFFFFF"], [1, COLORES[0]]], showscale=False), row=1, col=col)
    fig.update_xaxes(title_text="predicha", row=1, col=col)
    fig.update_yaxes(title_text="real", autorange="reversed", row=1, col=col)
fig.update_layout(title="Matrices de confusión en test", height=420)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Número de falsos positivos y falsos negativos de cada modelo y si el error es simétrico.

### 21.4 Desempeño según el largo de la reseña

Se compara el F1 macro de test de ambos modelos finales en los tres tramos de largo. Las reseñas de más de 320 tokens se leyeron truncadas.

In [ ]:
# ============================================================
# F1 de test por tramo de largo
# ============================================================
tabla_tramos_test = pd.concat([f1_por_largo(p["y_true"], p["y_pred"], DATOS_TEST["largo_original"]).assign(modelo=familia)
                               for familia, p in pred_test.items()], ignore_index=True)
display(tabla_tramos_test.pivot(index="tramo", columns="modelo", values="f1_macro").loc[[t for _, _, t in TRAMOS]].round(4))

In [ ]:
# ============================================================
# Gráfico del F1 de test por tramo de largo
# ============================================================
fig = px.bar(tabla_tramos_test, x="tramo", y="f1_macro", color="modelo", barmode="group",
             color_discrete_sequence=[COLORES[0], COLORES[2]], category_orders={"tramo": [t for _, _, t in TRAMOS]},
             labels={"f1_macro": "F1 macro (test)", "tramo": "largo original de la reseña"}, height=420)
fig.update_layout(title="F1 macro de test por largo de la reseña")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Tramo en que cada modelo rinde peor.
- ¿Las reseñas truncadas pierden desempeño frente a las que se leyeron completas?

### 21.5 Análisis de errores

Se muestran los errores de test con mayor confianza del mejor modelo en test: reseñas en que asignó una probabilidad alta a la clase incorrecta. Permiten distinguir errores del modelo de reseñas ambiguas (por ejemplo, sarcasmo o reseñas que critican la película pero recomiendan verla).

In [ ]:
# ============================================================
# Errores con mayor confianza en test
# ============================================================
familia_mejor_test = max(pred_test, key=lambda f: f1_score(pred_test[f]["y_true"], pred_test[f]["y_pred"], average="macro"))
p = pred_test[familia_mejor_test]
confianza = p["probas"].max(1)
errores = np.where(p["y_pred"] != p["y_true"])[0]
peores = errores[np.argsort(-confianza[errores])][:8]
print(f"Modelo: {familia_mejor_test} | errores en test: {len(errores)} de {len(p['y_true'])} "
      f"| con confianza ≥ 0,9: {(confianza[errores] >= 0.9).sum()}")
display(pd.DataFrame({"real": [SENTIMIENTOS[p["y_true"][i]] for i in peores],
                      "predicha": [SENTIMIENTOS[p["y_pred"][i]] for i in peores],
                      "confianza": confianza[peores].round(3),
                      "inicio de la reseña": [imdb["review"].iloc[idx_test[i]][:300] for i in peores]}))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Cuántos errores tienen confianza alta.
- Patrones en las reseñas mal clasificadas (sarcasmo, contraste, opinión mixta, señal al final de una reseña truncada).

## 22. Comparación entre la red recurrente y el Transformer

Ambos modelos finales se entrenaron con los mismos datos, vocabulario, largo máximo, *embedding* de entrada, capa de representación, métricas y régimen de épocas, y cada uno con su propia optimización completa. Las diferencias entre ellos se deben entonces a la arquitectura.

### 22.1 Desempeño y costo

La tabla reúne, para cada familia, el F1 macro de validación con 5 semillas (media y desviación), el F1 de test, los parámetros y el tiempo por época. La diferencia de validación se compara con el mayor de los dos rangos de error.

In [ ]:
# ============================================================
# Red recurrente frente a Transformer
# ============================================================
comparacion = pd.DataFrame({
    familia: {"f1_val_media_5_semillas": semillas["f1_macro_val"].mean(), "f1_val_desviacion": semillas["f1_macro_val"].std(),
              "f1_test": f1_score(pred_test[familia]["y_true"], pred_test[familia]["y_pred"], average="macro"),
              "exactitud_test": accuracy_score(pred_test[familia]["y_true"], pred_test[familia]["y_pred"]),
              "parametros": fila["parametros"], "segundos_por_epoca": fila["segundos_por_epoca"],
              "mejor_epoca": fila["mejor_epoca"], "rango_de_error": margen}
    for familia, semillas, fila, margen in [("recurrente", tabla_final_semillas_rnn, fila_final_rnn, MARGEN_ERROR_RNN),
                                            ("Transformer", tabla_final_semillas_trf, fila_final_trf, MARGEN_ERROR_TRF)]
})
diferencia_val = comparacion.loc["f1_val_media_5_semillas", "Transformer"] - comparacion.loc["f1_val_media_5_semillas", "recurrente"]
print(f"Transformer − recurrente (validación, 5 semillas): {diferencia_val * 100:+.2f} pp "
      f"| mayor rango de error: {max(MARGEN_ERROR_RNN, MARGEN_ERROR_TRF) * 100:.2f} pp")
display(comparacion.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿La diferencia de validación entre familias supera el rango de error?
- Costo de cada familia (parámetros, segundos por época) frente a su desempeño.
- Tramos de largo (sección 21.4) en que una familia aventaja a la otra.

### 22.2 Diferencias de arquitectura y atención visualizada

| Aspecto | Red recurrente | Transformer |
|---|---|---|
| Cómo recorre la reseña | Token a token, en orden; cada paso depende del anterior | Todos los tokens a la vez |
| Distancia entre dos tokens | Tantos pasos como tokens los separen; la señal se atenúa | Un solo paso de atención, a cualquier distancia |
| Paralelismo en GPU | Limitado por la secuencia (cada paso espera al anterior) | Alto dentro de cada capa |
| Costo según el largo L | Crece con L | La atención crece con L² |
| Orden de las palabras | Implícito en la recurrencia | *Embedding* de posición aprendido |
| Interpretabilidad | El estado oculto no indica qué tokens usó | Los pesos de atención muestran a qué tokens mira |

Para el último punto, `pesos_atencion(modelo, ids)` activa el registro de la atención, procesa una reseña y retorna el peso de cada token: con el decoder, el peso de la atención cruzada de su consulta; con `[CLS]`, la atención del token `<cls>` en el último bloque; con el promedio, la atención media que recibe cada token en el último bloque. En todos los casos se promedian las cabezas. Se grafica para una reseña positiva y una negativa de validación, bien clasificadas, con los primeros 60 tokens.

In [ ]:
# ============================================================
# Pesos de atención por token
# ============================================================
def pesos_atencion(modelo, ids):
    modelo.eval()
    capas_registro = list(modelo.bloques) + ([modelo.decoder] if modelo.agregacion == "decoder" else [])
    for capa in capas_registro:
        capa.guardar_atencion = True
    with torch.no_grad():
        x = torch.tensor([ids], device=device)
        modelo(x, torch.tensor([len(ids)], device=device))
    for capa in capas_registro:
        capa.guardar_atencion = False
    if modelo.agregacion == "decoder":
        return modelo.decoder.atencion[0].mean(0)[0].cpu().numpy()
    atencion = modelo.bloques[-1].atencion[0].mean(0)
    if modelo.agregacion == "cls":
        return atencion[0, 1:].cpu().numpy()
    return atencion.mean(0).cpu().numpy()


aciertos_val = np.where(pred_val["Transformer"]["y_pred"] == y_val)[0]
ejemplos_atencion = {c: next((i for i in aciertos_val if y_val[i] == c and 40 <= DATOS_VAL["largos_cpu"][i] <= 120),
                             int(np.where(y_val == c)[0][0])) for c in [1, 0]}
print({SENTIMIENTOS[c]: int(i) for c, i in ejemplos_atencion.items()})

In [ ]:
# ============================================================
# Atención sobre una reseña positiva de validación
# ============================================================
i = ejemplos_atencion[1]
ids = DATOS_VAL["x"][i, :int(DATOS_VAL["largos_cpu"][i])].tolist()
pesos = pesos_atencion(modelo_final_trf, ids)[:60]
tokens = [VOCAB.get_itos()[t] for t in ids[:60]]
fig = go.Figure(go.Bar(x=list(range(len(tokens))), y=pesos, marker_color=COLORES[0]))
fig.update_xaxes(tickvals=list(range(len(tokens))), ticktext=tokens, tickangle=-60)
fig.update_layout(title=f"Peso de atención por token · reseña positiva (agregación: {modelo_final_trf.agregacion})",
                  yaxis_title="peso de atención", height=460)
fig.show()

In [ ]:
# ============================================================
# Atención sobre una reseña negativa de validación
# ============================================================
i = ejemplos_atencion[0]
ids = DATOS_VAL["x"][i, :int(DATOS_VAL["largos_cpu"][i])].tolist()
pesos = pesos_atencion(modelo_final_trf, ids)[:60]
tokens = [VOCAB.get_itos()[t] for t in ids[:60]]
fig = go.Figure(go.Bar(x=list(range(len(tokens))), y=pesos, marker_color=COLORES[0]))
fig.update_xaxes(tickvals=list(range(len(tokens))), ticktext=tokens, tickangle=-60)
fig.update_layout(title=f"Peso de atención por token · reseña negativa (agregación: {modelo_final_trf.agregacion})",
                  yaxis_title="peso de atención", height=460)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Tokens con mayor peso en cada reseña: ¿son palabras de sentimiento, negaciones o signos?
- Si el peso se concentra en pocos tokens o se reparte.

### 22.3 BLEU y ROUGE

BLEU y ROUGE miden cuántos n-gramas de un **texto generado** coinciden con un texto de referencia escrito por personas; se usan en traducción, resumen y generación de diálogo. Este problema es de **clasificación**: el modelo no genera texto, sino una de dos etiquetas, y no existe un texto de referencia con el que comparar. Por eso no se calculan. Las métricas que corresponden a una salida categórica son las de las secciones 21 y 22.1 (exactitud, precisión, exhaustividad y F1).

## 23. Comparación entre las configuraciones base y finales

Se muestran los hiperparámetros que cambiaron en cada familia. El aporte de cada decisión está en las tablas de decisiones (secciones 13.1 y 20.1) y la comparación con 5 semillas, en las secciones 13.4 y 20.4.

In [ ]:
# ============================================================
# Configuración base frente a final (recurrente)
# ============================================================
config_base_vs_final_rnn = pd.DataFrame({"base": pd.Series(CONFIG_BASE_RNN).astype(str), "final": pd.Series(CONFIG_FINAL_RNN).astype(str)})
config_base_vs_final_rnn["cambia"] = config_base_vs_final_rnn["base"] != config_base_vs_final_rnn["final"]
display(config_base_vs_final_rnn)

In [ ]:
# ============================================================
# Configuración base frente a final (Transformer)
# ============================================================
config_base_vs_final_trf = pd.DataFrame({"base": pd.Series(CONFIG_BASE_TRF).astype(str), "final": pd.Series(CONFIG_FINAL_TRF).astype(str)})
config_base_vs_final_trf["cambia"] = config_base_vs_final_trf["base"] != config_base_vs_final_trf["final"]
display(config_base_vs_final_trf)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Hiperparámetros que cambiaron en cada familia y cuáles de esas decisiones superan su rango de error.
- Decisiones tomadas en empate técnico y su criterio.

## 24. Síntesis de los experimentos

Tabla con todas las variantes entrenadas o recuperadas de ambas familias, en el orden de los experimentos.

In [ ]:
# ============================================================
# Resumen de todas las variantes
# ============================================================
resumen_general = pd.concat([t.assign(familia=c["nombre"]) for c in (CADENA_RNN, CADENA_TRF) for t in c["resultados"]],
                            ignore_index=True)
display(resumen_general[["familia", "experimento"] + COLUMNAS_TABLA].round(4))

## 25. Modelo de texto para la fusión

Al notebook de fusión pasa la representación de 128 valores del modelo con **mejor F1 macro de validación**, medido como la media de las 5 semillas de la configuración final (secciones 13.4 y 20.4), que es más estable que una sola corrida. Si la diferencia no supera el mayor de los rangos de error, la elección es un empate técnico y se decide con el usuario.

In [ ]:
# ============================================================
# Selección del modelo de texto para la fusión
# ============================================================
FAMILIA_FUSION = comparacion.loc["f1_val_media_5_semillas"].idxmax()
# FAMILIA_FUSION = "<recurrente o Transformer>"  # Cambio Manual
MODELO_FUSION = {"recurrente": modelo_final_rnn, "Transformer": modelo_final_trf}[FAMILIA_FUSION]
CONFIG_FUSION = {"recurrente": CONFIG_FINAL_RNN, "Transformer": CONFIG_FINAL_TRF}[FAMILIA_FUSION]
print(f"Modelo para la fusión: {FAMILIA_FUSION} | diferencia de validación: {diferencia_val * 100:+.2f} pp")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Modelo elegido y si la diferencia con la otra familia supera el rango de error.
- Si hay empate técnico, opciones y criterio (se decide con el usuario).

### 25.1 Representación del texto

Se extrae la representación de las reseñas de validación con el modelo elegido y se proyecta a dos dimensiones con **PCA** (análisis de componentes principales), coloreada por sentimiento, para comprobar que el vector separa las clases.

`extraer_representaciones(modelo, datos)` aplica `representacion()` por lotes ordenados por largo, sin gradientes, y retorna un arreglo `[reseñas, 128]` en el orden original.

In [ ]:
# ============================================================
# Extracción de representaciones y proyección PCA
# ============================================================
def extraer_representaciones(modelo, datos, lote=512):
    modelo.eval()
    orden = torch.argsort(datos["largos_cpu"], descending=True).to(device)
    salida = torch.zeros(len(orden), UNIDADES_REPRESENTACION, device=device)
    with torch.no_grad():
        for inicio in range(0, len(orden), lote):
            indices = orden[inicio:inicio + lote]
            largos = datos["largos"][indices]
            salida[indices] = modelo.representacion(datos["x"][indices, :int(largos.max())], largos)
    return salida.cpu().numpy()


representaciones_val = extraer_representaciones(MODELO_FUSION, DATOS_VAL)
pca = PCA(n_components=2, random_state=SEED)
proyeccion = pca.fit_transform(representaciones_val)
print(f"Forma: {representaciones_val.shape} | varianza explicada por 2 componentes: {pca.explained_variance_ratio_.sum() * 100:.1f} %")

In [ ]:
# ============================================================
# Proyección de la representación por sentimiento
# ============================================================
fig = px.scatter(x=proyeccion[:, 0], y=proyeccion[:, 1], color=[SENTIMIENTOS[c] for c in y_val], opacity=0.5,
                 color_discrete_sequence=[COLORES[2], COLORES[0]],
                 labels={"x": "componente 1", "y": "componente 2", "color": "sentimiento"}, height=520)
fig.update_traces(marker_size=4)
fig.update_layout(title=f"Representación del texto en validación (modelo {FAMILIA_FUSION}, proyección PCA)")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Las reseñas positivas y negativas forman grupos separados? ¿Cuánto se superponen?

### 25.2 Prueba con reseñas escritas

Para la ejecución en vivo, `predecir_resena(modelo, texto)` limpia, tokeniza y codifica un texto cualquiera con el mismo vocabulario, y retorna la probabilidad de que sea positivo. Se prueba con frases cortas que incluyen negaciones y contrastes, donde el orden de las palabras cambia el sentido.

In [ ]:
# ============================================================
# Predicción de reseñas escritas
# ============================================================
def predecir_resena(modelo, texto):
    ids = [VOCAB[t] for t in tokenizar(texto)[:MAXLEN]] or [UNK_IDX]
    modelo.eval()
    with torch.no_grad():
        logits = modelo(torch.tensor([ids], device=device), torch.tensor([len(ids)], device=device))
    return torch.softmax(logits, 1)[0, 1].item()


RESENAS_PRUEBA = ["This movie was great, I loved every minute of it!",
                  "Not good. The plot was boring and the acting was terrible.",
                  "Not good, but fun.",
                  "Not fun, but good.",
                  "I expected a masterpiece and got a waste of two hours."]
display(pd.DataFrame({"reseña": RESENAS_PRUEBA,
                      "P(positiva) recurrente": [predecir_resena(modelo_final_rnn, t) for t in RESENAS_PRUEBA],
                      "P(positiva) Transformer": [predecir_resena(modelo_final_trf, t) for t in RESENAS_PRUEBA]}).round(3))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Los modelos distinguen «Not good, but fun» de «Not fun, but good»?
- Frases en que ambos modelos discrepan.

## 26. Conclusiones generales

*(completar después de ejecutar, con las cifras obtenidas: desempeño de cada familia frente a las líneas base, efecto del tipo de celda y de la forma de resumir, hiperparámetros de entrenamiento, comparación entre familias, decisiones en empate y limitaciones)*

### 26.1 Mejoras propuestas

*(completar después de ejecutar)*

## 27. Conclusión final

*(completar después de ejecutar)*

## 28. Referencias

- Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H. y Bengio, Y. (2014). *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*. EMNLP. arXiv:1406.1078.
- Devlin, J., Chang, M.-W., Lee, K. y Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL. arXiv:1810.04805.
- Hochreiter, S. y Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation, 9(8), 1735–1780.
- Kingma, D. P. y Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR. arXiv:1412.6980.
- Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y. y Potts, C. (2011). *Learning Word Vectors for Sentiment Analysis*. ACL.
- Pascanu, R., Mikolov, T. y Bengio, Y. (2013). *On the difficulty of training recurrent neural networks*. ICML. arXiv:1211.5063.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. y Salakhutdinov, R. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR, 15, 1929–1958.
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł. y Polosukhin, I. (2017). *Attention Is All You Need*. NeurIPS. arXiv:1706.03762.
- PyTorch. *Documentación oficial*. https://pytorch.org/docs/stable/
- *EFT125_DLY0100_Estudiante* y *ET_Ejemplo_sentimiento_junto_clasificacion*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).

## 29. Anexo A — Síntesis visual para la presentación

Gráficos en formato 16:9 con la paleta de la presentación, generados desde las variables ya calculadas (no entrenan). Cada uno se exporta a HTML en `respaldo_eft_texto/graficos/`; el PNG se obtiene con el botón de cámara del gráfico o directamente si está instalado `kaleido`.

`estilo_presentacion(fig, titulo, subtitulo)` aplica el estilo común y retorna la figura. `exportar(fig, nombre)` guarda el gráfico en HTML (y en PNG si es posible).

In [ ]:
# ============================================================
# Estilo y exportación de los gráficos de la presentación
# ============================================================
RUTA_RESPALDO = "respaldo_eft_texto"
RUTA_GRAFICOS = f"{RUTA_RESPALDO}/graficos"
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
AZUL_MARINO = "#0E2747"
ANCHO_PNG, ALTO_PNG = 1600, 900


def estilo_presentacion(fig, titulo, subtitulo=None):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:15px;color:#555555'>{subtitulo}</span>"
    fig.update_layout(template="plotly_white", title={"text": texto, "x": 0.02},
                      font={"family": "Arial", "size": 16, "color": AZUL_MARINO},
                      height=560, margin={"t": 110, "l": 80, "r": 40, "b": 80},
                      legend={"orientation": "h", "y": -0.18, "title": None})
    return fig


def exportar(fig, nombre):
    fig.write_html(f"{RUTA_GRAFICOS}/{nombre}.html", include_plotlyjs="cdn",
                   config={"toImageButtonOptions": {"format": "png", "width": ANCHO_PNG, "height": ALTO_PNG,
                                                    "scale": 2, "filename": nombre}})
    try:
        fig.write_image(f"{RUTA_GRAFICOS}/{nombre}.png", width=ANCHO_PNG, height=ALTO_PNG, scale=2)
        print(f"{nombre}: guardado en HTML y PNG")
    except Exception:
        print(f"{nombre}: guardado en HTML (PNG desde el botón de cámara del gráfico)")

### A.1 Tipo de celda según el largo de la reseña

In [ ]:
# ============================================================
# A.1 Tipo de celda según el largo de la reseña
# ============================================================
fig = px.bar(tabla_tramos_celda, x="tramo", y="f1_macro", color="celda", barmode="group",
             color_discrete_sequence=COLORES, category_orders={"tramo": [t for _, _, t in TRAMOS]},
             text=tabla_tramos_celda["f1_macro"].map(lambda v: f"{v:.3f}"))
fig.update_xaxes(title_text="largo original de la reseña")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Las compuertas sostienen la señal en reseñas largas", "F1 macro de validación por tramo de largo")
exportar(fig, "a1_celda_por_largo")
fig.show()

### A.2 Forma de resumir la reseña en el Transformer

In [ ]:
# ============================================================
# A.2 Promedio, [CLS] y decoder
# ============================================================
datos = tabla_de(CADENA_TRF, "agregación")
fig = go.Figure(go.Bar(x=datos["variante"], y=datos["f1_macro_val"], marker_color=COLORES[0],
                       text=[f"{v:.3f}<br>{p / 1e6:.2f} M parámetros" for v, p in zip(datos["f1_macro_val"], datos["parametros"])],
                       textposition="outside"))
fig.update_yaxes(title_text="F1 macro (validación)", range=[datos["f1_macro_val"].min() - 0.03, 1])
estilo_presentacion(fig, "Cómo resume el Transformer la reseña", "Encoder con promedio, token [CLS] o decoder con atención cruzada")
exportar(fig, "a2_agregacion")
fig.show()

### A.3 Tasa de aprendizaje en ambas familias

In [ ]:
# ============================================================
# A.3 Tasa de aprendizaje
# ============================================================
fig = go.Figure()
for cadena, color in [(CADENA_RNN, COLORES[0]), (CADENA_TRF, COLORES[2])]:
    datos = tabla_de(cadena, "tasa de aprendizaje").assign(tasa=lambda d: d["variante"].astype(float)).sort_values("tasa")
    fig.add_trace(go.Scatter(x=datos["tasa"], y=datos["f1_macro_val"], mode="lines+markers", name=cadena["nombre"],
                             line={"color": color, "width": 3}, marker_size=12))
fig.update_xaxes(title_text="tasa de aprendizaje", type="log")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Tasa de aprendizaje", "F1 macro de validación según la tasa, por familia")
exportar(fig, "a3_tasa")
fig.show()

### A.4 Épocas sin Early Stopping

In [ ]:
# ============================================================
# A.4 Pérdida de validación sin Early Stopping, ambas familias
# ============================================================
fig = go.Figure()
for nombre, historia, color in [("recurrente", historia_epocas_rnn, COLORES[0]), ("Transformer", historia_epocas_trf, COLORES[2])]:
    epocas = list(range(1, len(historia["loss_val"]) + 1))
    fig.add_trace(go.Scatter(x=epocas, y=historia["loss_val"], name=f"{nombre} (validación)", line={"color": color, "width": 3}))
    fig.add_trace(go.Scatter(x=epocas, y=historia["loss_train"], name=f"{nombre} (entrenamiento)",
                             line={"color": color, "width": 2, "dash": "dot"}))
fig.update_xaxes(title_text="época")
fig.update_yaxes(title_text="pérdida (entropía cruzada)")
estilo_presentacion(fig, "Dónde empieza el sobreajuste", "Pérdida por época sin Early Stopping")
exportar(fig, "a4_epocas")
fig.show()

### A.5 Modelos base y finales con 5 semillas

In [ ]:
# ============================================================
# A.5 Base frente a final con 5 semillas, por familia
# ============================================================
fig = go.Figure()
for nombre, tabla, color in [("Recurrente base", tabla_variabilidad_rnn, COLORES[2]), ("Recurrente final", tabla_final_semillas_rnn, COLORES[0]),
                             ("Transformer base", tabla_variabilidad_trf, COLORES[2]), ("Transformer final", tabla_final_semillas_trf, COLORES[0])]:
    fig.add_trace(go.Box(y=tabla["f1_macro_val"], name=nombre, marker_color=color, boxpoints="all", jitter=0.3))
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Configuraciones base frente a finales", "F1 macro de validación con las mismas 5 semillas")
exportar(fig, "a5_base_vs_final")
fig.show()

### A.6 Recurrente frente a Transformer en test

In [ ]:
# ============================================================
# A.6 Resultado en test frente a las líneas base
# ============================================================
fig = go.Figure(go.Bar(x=tabla_test["modelo"], y=tabla_test["f1_macro"],
                       marker_color=[COLORES[2]] * 3 + [COLORES[0]] * 2,
                       text=[f"{v:.3f}" for v in tabla_test["f1_macro"]], textposition="outside"))
fig.update_yaxes(title_text="F1 macro (test)", range=[0, 1.05])
estilo_presentacion(fig, "Resultado en test", "Modelos finales frente a las líneas base")
exportar(fig, "a6_test")
fig.show()

### A.7 Impacto de cada decisión

In [ ]:
# ============================================================
# A.7 Aporte de cada decisión, por familia
# ============================================================
datos = pd.concat([tabla_decisiones_rnn.assign(familia="recurrente", margen=MARGEN_ERROR_RNN),
                   tabla_decisiones_trf.assign(familia="Transformer", margen=MARGEN_ERROR_TRF)]).dropna(subset=["aporte"])
etiquetas = datos["familia"] + " · " + datos["experimento"]
fig = go.Figure(go.Bar(x=etiquetas, y=datos["aporte"] * 100,
                       marker_color=[COLORES[0] if s else COLORES[2] for s in datos["supera_margen"]],
                       text=[f"{v * 100:+.2f}" for v in datos["aporte"]], textposition="outside"))
fig.update_yaxes(title_text="aporte al F1 macro (pp)")
estilo_presentacion(fig, "Aporte de cada decisión", "Azul oscuro: supera el rango de error de su familia")
exportar(fig, "a7_decisiones")
fig.show()

## 30. Anexo B — Herramientas y tecnologías de apoyo utilizadas

| Categoría | Herramienta | Uso |
|---|---|---|
| Entorno de ejecución | Google Colab con GPU NVIDIA T4 | Entrenamiento de todos los modelos |
| Editor | Visual Studio Code con la extensión de Google Colab | Edición y ejecución del notebook conectado al servidor de Colab |
| Asistente de IA | Claude Opus 5.5 (Anthropic) | Estructuración del notebook, apoyo en el código y en la redacción; los resultados, decisiones e interpretaciones se revisaron con las salidas del propio notebook |
| Framework | PyTorch (https://pytorch.org) | Definición y entrenamiento de las redes recurrentes y del Transformer |
| Librerías | NumPy (https://numpy.org), Pandas (https://pandas.pydata.org), scikit-learn (https://scikit-learn.org), Plotly (https://plotly.com/python/) | Arreglos, tablas, métricas, referencia lineal, PCA y gráficos |
| Repositorio | GitHub | Datos, resultados guardados y versiones del notebook |

## 31. Respaldo de resultados

Se guardan en `respaldo_eft_texto/` y se comprimen en `respaldo_eft_texto.zip` todas las tablas, las historias y los modelos finales. El archivo `texto_final.pt` es el que usa el notebook de fusión: contiene la familia elegida, su configuración, sus pesos, el vocabulario y los índices de la partición. Se guarda con tensores y tipos básicos para que la fusión lo cargue con `weights_only=True`, sin ejecutar código serializado.

Con un servidor remoto (Colab), los archivos quedan en el servidor: hay que descargar el `.zip`, `texto_final.pt` y `resultados_texto_guardados.json` desde el panel de archivos antes de cerrar la sesión.

In [ ]:
# ============================================================
# Respaldo de resultados
# ============================================================
os.makedirs(RUTA_RESPALDO, exist_ok=True)
resumen_general.to_csv(f"{RUTA_RESPALDO}/experimentos_todas_las_ejecuciones.csv", index=False)
tabla_lineas_base_val.to_csv(f"{RUTA_RESPALDO}/lineas_base_validacion.csv", index=False)
tabla_maxlen.to_csv(f"{RUTA_RESPALDO}/cobertura_maxlen.csv", index=False)
tabla_vocab.to_csv(f"{RUTA_RESPALDO}/vocabulario_min_freq.csv", index=False)
tabla_tramos_celda.to_csv(f"{RUTA_RESPALDO}/f1_por_largo_celda.csv", index=False)
tabla_tramos_test.to_csv(f"{RUTA_RESPALDO}/f1_por_largo_test.csv", index=False)
tabla_test.to_csv(f"{RUTA_RESPALDO}/metricas_test.csv", index=False)
reporte.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase.csv")
comparacion.to_csv(f"{RUTA_RESPALDO}/recurrente_vs_transformer.csv")
for sufijo, tablas in {"rnn": [tabla_variabilidad_rnn, tabla_epocas_rnn, tabla_decisiones_rnn, tabla_verificacion_rnn,
                               tabla_final_semillas_rnn, comparacion_justa_rnn, config_base_vs_final_rnn],
                       "trf": [tabla_variabilidad_trf, tabla_epocas_trf, tabla_decisiones_trf, tabla_verificacion_trf,
                               tabla_final_semillas_trf, comparacion_justa_trf, config_base_vs_final_trf]}.items():
    for nombre, tabla in zip(["variabilidad", "epocas_sin_early_stopping", "decisiones", "verificaciones_orden",
                              "final_5_semillas", "comparacion_base_final", "config_base_vs_final"], tablas):
        tabla.to_csv(f"{RUTA_RESPALDO}/{nombre}_{sufijo}.csv")
with open(f"{RUTA_RESPALDO}/historias.json", "w") as f:
    json.dump({"base_rnn": historia_base_rnn, "final_rnn": historia_final_rnn, "sin_es_rnn": historia_epocas_rnn,
               "base_trf": historia_base_trf, "final_trf": historia_final_trf, "sin_es_trf": historia_epocas_trf,
               "celda": historias_celda, "agregacion": historias_agregacion}, f)
for familia, p in pred_test.items():
    np.save(f"{RUTA_RESPALDO}/y_test_proba_{familia}.npy", p["probas"])
particion = {"idx_train": torch.tensor(idx_train), "idx_val": torch.tensor(idx_val), "idx_test": torch.tensor(idx_test)}
for familia, modelo, config in [("recurrente", modelo_final_rnn, CONFIG_FINAL_RNN), ("Transformer", modelo_final_trf, CONFIG_FINAL_TRF)]:
    torch.save({"familia": familia, "config": config, "estado": modelo.state_dict(), "itos": VOCAB.get_itos(), **particion},
               f"{RUTA_RESPALDO}/{familia.lower()}_final.pt")
torch.save({"familia": FAMILIA_FUSION, "config": CONFIG_FUSION, "estado": MODELO_FUSION.state_dict(),
            "itos": VOCAB.get_itos(), **particion}, "texto_final.pt")
shutil.copy("texto_final.pt", f"{RUTA_RESPALDO}/texto_final.pt")
guardar_corridas()
shutil.copy(ARCHIVO_RESULTADOS, f"{RUTA_RESPALDO}/{ARCHIVO_RESULTADOS}")
shutil.make_archive(RUTA_RESPALDO, "zip", RUTA_RESPALDO)
print(f"Respaldo guardado en {RUTA_RESPALDO}.zip ({os.path.getsize(RUTA_RESPALDO + '.zip') / 1e6:.1f} MB) "
      f"| texto_final.pt: {os.path.getsize('texto_final.pt') / 1e6:.1f} MB")

## 32. Descripción de las salidas

| Archivo | Contenido | Uso posterior |
|---|---|---|
| `resultados_texto_guardados.json` | Métricas de cada corrida, por clave de configuración y semilla | Subirlo a la raíz del repositorio para que la siguiente ejecución no repita corridas |
| `texto_final.pt` | Familia elegida, configuración, pesos, vocabulario e índices de la partición del modelo de texto para la fusión | Subirlo a la raíz del repositorio: lo carga el notebook de fusión |
| `respaldo_eft_texto/recurrente_final.pt`, `transformer_final.pt` | Modelos finales de ambas familias | Reutilizar el modelo no elegido sin reentrenar |
| `respaldo_eft_texto/*.csv` | Tablas de experimentos, variabilidad, decisiones, verificaciones, test, desempeño por largo y comparación entre familias | Regenerar tablas y gráficos sin reentrenar |
| `respaldo_eft_texto/historias.json` | Curvas por época de los modelos base, finales, sin Early Stopping, de las celdas y de las formas de resumir | Gráficos de la presentación |
| `respaldo_eft_texto/y_test_proba_*.npy` | Probabilidades de cada modelo final en test | Análisis de errores |
| `respaldo_eft_texto/graficos/` | Gráficos del anexo A en HTML (y PNG si hay `kaleido`) | Diapositivas de la presentación |
| `respaldo_eft_texto.zip` | Todo lo anterior comprimido | Descarga desde Colab |